# 第62课：PPO 中的 Actor/Critic 联合目标、Value Clipping 与 GAE 工程集成

**把第60课的策略裁剪和第61课的价值估计真正接成一个训练批次。**

这次把一批真实 on-policy 轨迹冻结下来，用同一批数据进行有限次 PPO-Clip Actor 更新，以及 Critic 的固定 λ-return 回归。比较三种价值损失，核对梯度归属、截断、自举、旧值快照、KL 停止与重新采样。

**交付方式：** 本 Jupyter Notebook 与同名 `.py` 脚本完全独立，代码都在各自文件里。Python 3.10+、现有 PyTorch、CPU、离线，不安装 Gymnasium，也不下载语言模型。训练的是 3 状态、2 动作的合成环境，不是预训练 LLM 或完整版 PPO 基准。

建议两次学习：第一次完成 Value clipping 的手算与梯度；第二次完成 Buffer、联合优化器、重载与独立练习。

## 学习目标

完成后，应能够在代码与日志里区分：

1. PPO Actor 的 `min(rA,clip(r)A)` 与 Actor 熵正则。
2. Critic 的 **unclipped MSE**、**保守 max-squared value clipping** 与**仅裁剪预测值再做 MSE**，说明它们不是同一个目标。
3. 采样时的 `old_logp`、`old_values`，以及由 **旧 Critic** 计算并冻结的 GAE 与 λ-return。
4. 外部时间截断的 **bootstrap 保留**，但 GAE 不跨越回合边界。
5. KL 停止 Actor 后为何仍可以继续回归 Critic；为什么这不构成硬 KL 约束。

数据与模型都在本机生成；不调用任何 API、模型仓库或外部身份服务。下面的论文链接仅用于核对规范和算法约定。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第62课：PPO中Actor/Critic联合目标、Value Clipping与GAE工程集成。

Python 3.10+，仅PyTorch与标准库，离线CPU真实小模型训练。
三种价值损失方案共享PPO-Clip actor、冻结的GAE目标和同一采样调度。
这是3状态2动作的合成Actor-Critic，不是预训练LLM或PPO标准实现认证。
"""
from __future__ import annotations


import argparse
import copy
import hashlib
import itertools
import json
import math
import platform
import random
import sys
import tempfile
from collections import Counter
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit("需要现有PyTorch环境；选择与Notebook相同的Python解释器。") from exc

VERSION='lesson62-ppo-actor-critic-v1'
SOURCES={
    'ppo_paper':'https://arxiv.org/abs/1707.06347',
    'spinningup_ppo':'https://spinningup.openai.com/en/latest/algorithms/ppo.html',
    'spinningup_code':'https://spinningup.openai.com/en/latest/_modules/spinup/algos/pytorch/ppo/ppo.html',
    'sb3_ppo':'https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html',
    'sb3_implementation':'https://github.com/DLR-RM/stable-baselines3/blob/master/stable_baselines3/ppo/ppo.py',
    'gae':'https://arxiv.org/abs/1506.02438',
    'truncation':'https://gymnasium.farama.org/v1.1.1/tutorials/gymnasium_basics/handling_time_limits/',
    'torch_autograd':'https://docs.pytorch.org/docs/stable/notes/autograd.html',
}

class LessonError(ValueError):
    """本课程协议错误；不静默修改任务定义。"""

def require(condition, message):
    if not bool(condition):
        raise LessonError(message)

def canonical(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)

def sha256(value):
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()

def tensor_hash(named):
    h = hashlib.sha256()
    for name, tensor in sorted(named.items()):
        t = tensor.detach().cpu().contiguous()
        h.update(canonical([name, list(t.shape), str(t.dtype)]).encode())
        h.update(bytes(t.reshape(-1).view(torch.uint8).tolist()))
    return h.hexdigest()

def seed_all(seed):
    random.seed(seed)
    torch.manual_seed(seed)

@dataclass(frozen=True)
class Config:
    seed: int = 6201
    gamma: float = 0.95
    gae_lambda: float = 0.95
    horizon: int = 6
    updates: int = 45
    episodes_per_update: int = 24
    update_epochs: int = 4
    actor_lr: float = 0.025
    critic_lr: float = 0.018
    actor_hidden: int = 24
    critic_hidden: int = 24
    clip_ratio: float = 0.2
    clip_value: float = 0.2
    value_weight: float = 0.5
    entropy_coef: float = 0.01
    target_state_kl: float = 0.04
    grad_clip: float = 1.0
    eval_episodes: int = 120

    def validate(self):
        for name in ('seed','horizon','updates','episodes_per_update','update_epochs',
                     'actor_hidden','critic_hidden','eval_episodes'):
            value=getattr(self,name)
            require(type(value) is int and value >= (0 if name=='seed' else 1),name+'非法')
        require(3<=self.horizon<=16 and 1<=self.update_epochs<=8,'horizon或更新轮数超出教学范围')
        for name in ('gamma','gae_lambda','actor_lr','critic_lr','clip_ratio','clip_value',
                     'value_weight','entropy_coef','target_state_kl','grad_clip'):
            value=getattr(self,name)
            require(type(value) in (float,int) and math.isfinite(value),name+'必须有限数值')
        require(0<self.gamma<1 and 0<=self.gae_lambda<=1,'折扣/λ非法')
        require(self.actor_lr>0 and self.critic_lr>0 and 0<self.clip_ratio<1
                and self.clip_value>0 and self.value_weight>0 and self.entropy_coef>=0
                and self.target_state_kl>0 and self.grad_clip>0,'loss/裁剪参数非法')

In [2]:
torch.set_num_threads(1)
seed_all(6201)
print('Python:',platform.python_version(),'PyTorch:',torch.__version__)
print('默认设备:CPU；不访问网络')
print('完整训练默认计划:',asdict(Config()))

Python: 3.13.5 PyTorch: 2.10.0+cpu
默认设备:CPU；不访问网络
完整训练默认计划: {'seed': 6201, 'gamma': 0.95, 'gae_lambda': 0.95, 'horizon': 6, 'updates': 45, 'episodes_per_update': 24, 'update_epochs': 4, 'actor_lr': 0.025, 'critic_lr': 0.018, 'actor_hidden': 24, 'critic_hidden': 24, 'clip_ratio': 0.2, 'clip_value': 0.2, 'value_weight': 0.5, 'entropy_coef': 0.01, 'target_state_kl': 0.04, 'grad_clip': 1.0, 'eval_episodes': 120}


## 1. 固定三状态环境，不把截断当真正终止

位置为 0、1、2；达到位置3才是真正终止。动作0等待，动作1前进。每步成本−0.01，抵达目标额外奖励+1。

底层过程可以持续运行。为了收集长度有限的轨迹，外部设定最多6步。如果第6步还没有抵达目标，标记 `truncated=True`，**并不表示底层 MDP 进入终止状态**。

这决定了 TD residual：

\[
\delta_t=r_t+\gamma(1-\text{terminated}_t)V_{old}(s_{t+1})-V_{old}(s_t)
\]

真实终止时尾值是0；外部截断时要保留 `V_old(next_obs)`。但轨迹结束后不能把下一个新回合的优势接进当前回合。

[Gymnasium：时间限制与截断](https://gymnasium.farama.org/v1.1.1/tutorials/gymnasium_basics/handling_time_limits/)

In [3]:
GOAL=3
ACTION_WAIT, ACTION_ADVANCE=0,1
STEP_COST,SUCCESS_BONUS=-.01,1.0

@dataclass(frozen=True)
class Step:
    obs: int
    action: int
    reward: float
    next_obs: int
    terminated: bool
    truncated: bool
    old_logp: float

def validate_step(step):
    require(isinstance(step, Step), "Step类型错误")
    require(type(step.obs) is int and 0 <= step.obs < GOAL, "当前obs必须是非终止位置")
    require(type(step.next_obs) is int and 0 <= step.next_obs <= GOAL, "next_obs无效")
    require(type(step.action) is int and step.action in (ACTION_WAIT, ACTION_ADVANCE), "动作非法")
    require(type(step.terminated) is bool and type(step.truncated) is bool,
            "terminated/truncated必须是bool")
    require(not (step.terminated and step.truncated), "真实终止与外部截断在本课程中互斥")
    require(step.next_obs == min(step.obs + step.action, GOAL), "环境动态不匹配")
    require(step.terminated == (step.next_obs == GOAL), "终止状态不匹配")
    expected_reward = STEP_COST + (SUCCESS_BONUS if step.terminated else 0)
    require(math.isfinite(step.reward) and abs(step.reward - expected_reward) < 1e-10,
            "奖励必须来自固定环境，而不是模型自报")
    require(type(step.old_logp) is float and math.isfinite(step.old_logp) and step.old_logp <= 1e-6,
            "需要有效的行为log概率")

def validate_episode(episode, horizon):
    require(type(episode) is list and 1 <= len(episode) <= horizon, "episode长度错误")
    require(episode[0].obs == 0, "每回合必须从位置0开始")
    for i, step in enumerate(episode):
        validate_step(step)
        if i < len(episode) - 1:
            require(not step.terminated and not step.truncated, "回合结束后仍有步骤")
            require(episode[i + 1].obs == step.next_obs, "下一步观察不连续")
    require(episode[-1].terminated or episode[-1].truncated, "回合尾部必须有明确终止状态")
    require(not episode[-1].truncated or len(episode) == horizon,
            "外部时间上限只能在约定horizon触发")

def transition(pos, action, t, horizon):
    require(type(pos) is int and 0 <= pos < GOAL, "pos非法")
    require(type(action) is int and action in (0, 1), "action非法")
    require(type(t) is int and 0 <= t < horizon, "时间步非法")
    nxt = min(pos + action, GOAL)
    terminated = nxt == GOAL
    truncated = t + 1 >= horizon and not terminated
    reward = STEP_COST + (SUCCESS_BONUS if terminated else 0.)
    return nxt, reward, terminated, truncated

def checked_numeric(values, name):
    require(isinstance(values, (list, tuple)) and len(values) > 0, name + "必须为非空序列")
    require(all(type(x) in (int, float) and math.isfinite(x) for x in values), name + "必须有限")

def discounted_returns(rewards, gamma, bootstrap=0.0):
    """时间倒序累计，最后一个非终止cut使用V(final_obs)作为尾部。"""
    checked_numeric(rewards, "rewards")
    require(type(gamma) in (int, float) and 0 < gamma <= 1, "gamma非法")
    require(type(bootstrap) in (int, float) and math.isfinite(bootstrap), "bootstrap非法")
    acc = float(bootstrap)
    output = [0.] * len(rewards)
    for i in range(len(rewards)-1, -1, -1):
        acc = float(rewards[i]) + gamma*acc
        output[i] = acc
    return output

def generalized_advantage_estimation(episodes, values, next_values, gamma, lam):
    """独立episode：delta=r+gamma*(1-terminated)V(next)-V(s)。

    截断时delta保留V(final_obs)，但由于下一条记录是新回合，trace递推停止。
    自然rollout数组末尾也停止trace，若未终止则必须保留下一状态bootstrap。
    返回 A 与 lambda-return=A+V_old，不连接critic梯度。
    """
    require(type(episodes) is list and bool(episodes), "需要回合列表")
    require(type(gamma) in (float, int) and 0 < gamma <= 1, "gamma非法")
    require(type(lam) in (float, int) and 0 <= lam <= 1, "lambda非法")
    count = sum(len(ep) for ep in episodes)
    checked_numeric(values, "values")
    checked_numeric(next_values, "next_values")
    require(len(values) == len(next_values) == count, "值函数数组长度与步骤数量不符")
    advantages, targets, deltas = [], [], []
    offset = 0
    for ep in episodes:
        require(len(ep) > 0, "空回合")
        for j, step in enumerate(ep):
            validate_step(step)
            if j < len(ep)-1:
                require(not (step.terminated or step.truncated), "episode结束后仍有下一条步骤")
                require(ep[j+1].obs == step.next_obs, "GAE回合内部不连续")
        local_values = values[offset:offset + len(ep)]
        local_next = next_values[offset:offset + len(ep)]
        offset += len(ep)
        local_d = [step.reward + gamma*(0. if step.terminated else nv) - v
                   for step,v,nv in zip(ep,local_values,local_next)]
        local_adv = [0.] * len(ep)
        ahead = 0.
        for j in range(len(ep)-1, -1, -1):
            # 回合结束或采样切断后，不能把下一回合的优势连接过来。
            boundary = ep[j].terminated or ep[j].truncated
            ahead = local_d[j] + gamma*lam*(0. if boundary else ahead)
            local_adv[j] = ahead
        deltas.extend(local_d)
        advantages.extend(local_adv)
        targets.extend([a+v for a,v in zip(local_adv,local_values)])
    return {"delta":deltas, "advantage":advantages, "lambda_target":targets}

def exact_policy_values(forward_probabilities, gamma=0.95):
    """Bellman: 在底层无限时域、位置马尔可夫过程里计算精确V^pi。

    外部6步截断不属于底层MDP，所以不能把它写进该值函数。
    """
    checked_numeric(forward_probabilities, "forward_probabilities")
    require(len(forward_probabilities) == GOAL, "每个非终止位置一个前进概率")
    require(all(0 <= v <= 1 for v in forward_probabilities), "动作概率不在0..1")
    require(type(gamma) in (int,float) and 0 < gamma < 1, "无穷时域要求gamma小于1")
    values = [0.] * GOAL
    for s in range(GOAL-1, -1, -1):
        p = float(forward_probabilities[s])
        forward_value = SUCCESS_BONUS + STEP_COST if s == GOAL-1 else STEP_COST + gamma*values[s+1]
        values[s] = (p*forward_value + (1-p)*STEP_COST)/(1 - gamma*(1-p))
    return values

def finite_horizon_success_probability(forward_probabilities, horizon):
    """固定外部horizon内到达目标的精确概率，不等于底层无限时域V。"""
    require(type(horizon) is int and horizon >= 1, "horizon非法")
    checked_numeric(forward_probabilities, "policy")
    require(len(forward_probabilities)==GOAL and all(0 <= x <= 1 for x in forward_probabilities),
            "策略概率非法")
    active = [1., 0., 0.]
    finished = 0.
    for _ in range(horizon):
        new = [0.] * GOAL
        for s,pop in enumerate(active):
            prob = forward_probabilities[s]
            new[s] += pop*(1-prob)
            if s == GOAL-1:
                finished += pop*prob
            else:
                new[s+1] += pop*prob
        active = new
    require(abs(finished+sum(active)-1)<1e-9, "总概率不归一")
    return finished

def exact_time_limit_bias(forward_probs=(.5,.5,.5),gamma=.95,horizon=6):
    """精确枚举2^horizon个独立动作。只算位置0起点的截断自举偏差。"""
    require(type(horizon) is int and 1<=horizon<=12, "枚举horizon过大")
    true_v=exact_policy_values(list(forward_probs),gamma)
    require(all(0 < p < 1 for p in forward_probs), "枚举的每个动作必须有非零概率")
    expected_good=0.; expected_wrong=0.; covered_probability=0.; truncated_probability=0.
    samples=[]
    for bits in __import__('itertools').product((0,1),repeat=horizon):
        pos=0; probability=1.;ep=[]
        for t,action in enumerate(bits):
            probability*=forward_probs[pos] if action else 1-forward_probs[pos]
            nxt,reward,terminated,truncated=transition(pos,action,t,horizon)
            ep.append(Step(pos,action,reward,nxt,terminated,truncated,math.log(forward_probs[pos] if action else 1-forward_probs[pos])))
            if terminated or truncated:break
            pos=nxt
        # 对在终点前提前终止的序列，只计一次动作前缀，不能按完整bits重复累加。
        if ep[-1].terminated and bits[len(ep):] != (0,)*(horizon-len(ep)):
            continue
        # 退出前缀的概率已包含全部已执行动作；未使用位不参与概率。
        tail=0. if ep[-1].terminated else true_v[ep[-1].next_obs]
        good=discounted_returns([e.reward for e in ep],gamma,tail)[0]
        wrong=discounted_returns([e.reward for e in ep],gamma,0.)[0]
        covered_probability+=probability
        truncated_probability+=probability*ep[-1].truncated
        expected_good+=probability*good
        expected_wrong+=probability*wrong
        values=[true_v[e.obs] for e in ep]
        next_values=[0. if e.terminated else true_v[e.next_obs] for e in ep]
        samples.append((probability,ep,values,next_values))
    require(abs(covered_probability-1)<1e-9,"枚举覆盖概率不足")
    var_by_lambda={}
    for lam in (0.,.5,.95,1.):
        means=0.;second=0.
        for probability,ep,values,next_values in samples:
            a=generalized_advantage_estimation([ep],values,next_values,gamma,lam)['advantage'][0]
            means+=probability*a;second+=probability*a*a
        var_by_lambda[str(lam)]={"mean_advantage_at_start":means,
                                  "variance_at_start":max(0.,second-means*means)}
    return {"true_continuing_value_start":true_v[0],"truncation_probability":truncated_probability,
            "proper_bootstrap_expected_return":expected_good,
            "wrong_treat_time_limit_as_terminal_return":expected_wrong,
            "lost_expected_value":expected_good-expected_wrong,
            "lambda_gradient_noise_example":var_by_lambda,
            "scope":"exact_enumeration_fixed_policy_time_limit_outside_underlying_MDP"}

In [4]:
for t in [0,5]:
    print('在位置1等待，时间步',t,'→',transition(1,0,t,6))
print('位置2前进到目标→',transition(2,1,5,6))
info=exact_time_limit_bias(gamma=.95,horizon=6)
for key in ['true_continuing_value_start','proper_bootstrap_expected_return',
            'wrong_treat_time_limit_as_terminal_return','lost_expected_value']:
    print(key,info[key])

在位置1等待，时间步 0 → (1, -0.01, False, False)
在位置1等待，时间步 5 → (1, -0.01, False, True)
位置2前进到目标→ (3, 0.99, True, False)
true_continuing_value_start 0.7277399848828416
proper_bootstrap_expected_return 0.727739984882842
wrong_treat_time_limit_as_terminal_return 0.501697839453125
lost_expected_value 0.226042145429717


## 2. GAE 与 λ-return 必须在优化器更新之前冻结

给定旧价值估计与下一状态旧价值：

\[
\widehat A_t=\delta_t+\gamma\lambda\widehat A_{t+1}
\]

在真实终止、外部截断与采样片段末尾，优势递推停止。**仅在真正终止时停止下一状态的价值自举**。这是两个不同的门控。

本课 Critic 回归的目标为：

\[
\widehat V^{\lambda}_t=V_{old}(s_t)+\widehat A_t
\]

每批轨迹收集后，把 `old_values`、原始优势、归一化优势以及 λ-return 全部保存为不带梯度的张量。不能在第2次、第3次同批更新时重新计算，否则目标随新 Critic 漂移，已不是同一批固定目标。

这里选择 GAE-derived λ-return，**不是与 GAE 数值恒等的另一种 bootstrap-free Monte Carlo return**。[GAE 原论文](https://arxiv.org/abs/1506.02438)

In [5]:
# 使用两步演示：最后一步是外部截断，但delta仍包含下一状态的V。
ep=[Step(0,0,-.01,0,False,False,-.69),
    Step(0,0,-.01,0,False,True,-.69)]
values=[.6,.8]; next_values=[.8,.5]
for lam in [0.,.5,.95,1.]:
    result=generalized_advantage_estimation([ep],values,next_values,.9,lam)
    print('λ=',lam,'delta=',result['delta'],
          'advantages=',result['advantage'],'targets=',result['lambda_target'])

λ= 0.0 delta= [0.1100000000000001, -0.36000000000000004] advantages= [0.1100000000000001, -0.36000000000000004] targets= [0.7100000000000001, 0.44]
λ= 0.5 delta= [0.1100000000000001, -0.36000000000000004] advantages= [-0.051999999999999935, -0.36000000000000004] targets= [0.548, 0.44]
λ= 0.95 delta= [0.1100000000000001, -0.36000000000000004] advantages= [-0.19779999999999992, -0.36000000000000004] targets= [0.40220000000000006, 0.44]
λ= 1.0 delta= [0.1100000000000001, -0.36000000000000004] advantages= [-0.21399999999999997, -0.36000000000000004] targets= [0.386, 0.44]


## 3. 独立 Actor / Critic：联合训练目标不等于共享参数

Actor 接受状态 ID，返回等待与前进的两个 logits；Critic 返回一个价值估计。两个网络参数与优化器完全独立。

本课的联合损失是：

\[
L_{joint}=L_{PPO-Clip}-c_{ent}\,H(\pi)+c_V L_{value}
\]

本实现采用 **一次联合 backward，两个独立 optimizer**：Actor 参数只从 Actor loss 接受梯度，Critic 参数只从固定返回目标的 value loss 接受梯度。两侧是 disjoint parameter sets，不需要共享 backbone，也没有在此实现多头共享网络。

PPO-Clip 的原始解释与代码：[Spinning Up PPO](https://spinningup.openai.com/en/latest/algorithms/ppo.html)、[PyTorch 实现](https://spinningup.openai.com/en/latest/_modules/spinup/algos/pytorch/ppo/ppo.html)。

In [6]:
class Actor(nn.Module):
    def __init__(self, hidden=24):
        super().__init__()
        self.net=nn.Sequential(nn.Embedding(GOAL,hidden),nn.Linear(hidden,hidden),
                               nn.Tanh(),nn.Linear(hidden,2))

    def forward(self, states):
        require(isinstance(states,torch.Tensor) and states.dtype == torch.long,
                "actor状态必须是long Tensor")
        require(bool(((states >= 0)&(states < GOAL)).all()), "actor状态越界")
        return self.net(states)

class Critic(nn.Module):
    def __init__(self, hidden=24):
        super().__init__()
        self.net=nn.Sequential(nn.Embedding(GOAL,hidden),nn.Linear(hidden,hidden),
                               nn.Tanh(),nn.Linear(hidden,1))

    def forward(self, states):
        require(isinstance(states,torch.Tensor) and states.dtype == torch.long,
                "critic状态必须是long Tensor")
        require(bool(((states >= 0)&(states < GOAL)).all()), "critic状态越界")
        return self.net(states).squeeze(-1)

def actor_probabilities(actor):
    with torch.no_grad():
        actor.eval()
        probs=F.softmax(actor(torch.arange(GOAL,dtype=torch.long)),dim=-1)[:,ACTION_ADVANCE]
    return probs.tolist()

def value_prediction(critic):
    with torch.no_grad():
        critic.eval()
        return critic(torch.arange(GOAL,dtype=torch.long)).tolist()

def value_error(critic,actor,gamma):
    reference=exact_policy_values(actor_probabilities(actor),gamma)
    prediction=value_prediction(critic)
    return {"predicted":prediction,"exact_for_current_policy":reference,
            "rmse":math.sqrt(sum((a-b)**2 for a,b in zip(prediction,reference))/GOAL),
            "scope":"exact_infinite_horizon_value_of_current_policy_not_finite_time_limit_return"}

def collect_rollouts(actor, episodes, horizon, rng):
    require(type(episodes) is int and episodes >= 1, "episodes非法")
    require(type(horizon) is int and horizon >= 3, "horizon非法")
    result=[]
    actor.eval()
    with torch.no_grad():
        p=F.softmax(actor(torch.arange(GOAL,dtype=torch.long)),dim=-1)
    for _ in range(episodes):
        pos=0;ep=[]
        for t in range(horizon):
            action=1 if rng.random()<float(p[pos,1]) else 0
            logp=math.log(float(p[pos,action]))
            nxt,reward,terminated,truncated=transition(pos,action,t,horizon)
            ep.append(Step(pos,action,reward,nxt,terminated,truncated,logp))
            if terminated or truncated:
                break
            pos=nxt
        validate_episode(ep,horizon)
        result.append(ep)
    return result

def compute_targets(episodes,critic,gamma,lam,bootstrap_truncation=True):
    require(type(bootstrap_truncation) is bool, "bootstrap标志必须为bool")
    states=[s.obs for ep in episodes for s in ep]
    nextstates=[s.next_obs if not s.terminated else 0 for ep in episodes for s in ep]
    with torch.no_grad():
        values=critic(torch.tensor(states,dtype=torch.long)).tolist()
        nextvalues=critic(torch.tensor(nextstates,dtype=torch.long)).tolist()
    for i,step in enumerate(s for ep in episodes for s in ep):
        if step.terminated or (step.truncated and not bootstrap_truncation):
            nextvalues[i]=0.
    target=generalized_advantage_estimation(episodes,values,nextvalues,gamma,lam)
    return {"states":states,"next_states":nextstates,"values_old":values,
            "next_values_old":nextvalues,"advantages":target['advantage'],
            "lambda_returns":target['lambda_target'],"deltas":target['delta'],
            "old_logps":[s.old_logp for ep in episodes for s in ep],
            "actions":[s.action for ep in episodes for s in ep],
            "rewards":[s.reward for ep in episodes for s in ep],
            "terminated":[s.terminated for ep in episodes for s in ep],
            "truncated":[s.truncated for ep in episodes for s in ep],
            "episodes":len(episodes),"steps":len(states),
            "bootstrap_truncation":bootstrap_truncation,
            "scope":"on_policy_fixed_targets_detached_from_critic"}

def evaluate_policy(actor,critic,gamma,horizon,rollouts=200,seed=2026):
    prob=actor_probabilities(actor)
    exact_continuing=exact_policy_values(prob,gamma)
    exact_success=finite_horizon_success_probability(prob,horizon)
    sampled=collect_rollouts(actor,rollouts,horizon,random.Random(seed))
    success=sum(ep[-1].terminated for ep in sampled)
    truncated=sum(ep[-1].truncated for ep in sampled)
    return {"forward_probabilities":prob,"exact_continuing_value":exact_continuing,
            "exact_success_within_limit":exact_success,
            "sampled_successes":success,"sampled_episodes":rollouts,
            "sampled_success_rate":success/rollouts,"sampled_truncations":truncated,
            "value_rmse":value_error(critic,actor,gamma)['rmse'],
            "note":"固定环境的精确计算＋有限采样；并非模型数学推理能力"}

In [7]:
demo_actor=Actor(16)
demo_critic=Critic(16)
print('Actor参数:',sum(p.numel() for p in demo_actor.parameters()))
print('Critic参数:',sum(p.numel() for p in demo_critic.parameters()))
print('独立参数存储:',not {p.data_ptr() for p in demo_actor.parameters()} &
      {p.data_ptr() for p in demo_critic.parameters()})
print('当前策略前进概率:',actor_probabilities(demo_actor))

Actor参数: 354
Critic参数: 337
独立参数存储: True
当前策略前进概率: [0.4076926112174988, 0.7102592587471008, 0.6786260008811951]


## 4. Actor 的比例裁剪：不要漏掉 min

对当前样本动作，旧 log 概率在采样时固定，当前 log 概率可导：

\[
r_t(\theta)=\exp[\log\pi_{\theta}(a_t|s_t)-\log\pi_{old}(a_t|s_t)]
\]

PPO-Clip 需要**最大化**：

\[
\min\big(r_t A_t,\;\operatorname{clip}(r_t,1-\epsilon,1+\epsilon)A_t\big)
\]

本课最小化它的负平均，再减去熵系数乘平均 categorical entropy。旧策略概率与固定优势都要 `detach()`，新 log 概率必须保持梯度。

裁剪控制的是**单项目标的额外收益**，不是保证更新后的概率比不会越界。`clip_grad_norm_()` 则是参数梯度裁剪，两者完全不同。

In [8]:
def probability_ratio(new_logp, old_logp):
    require(isinstance(new_logp,torch.Tensor) and isinstance(old_logp,torch.Tensor)
            and new_logp.shape==old_logp.shape and new_logp.ndim==1 and new_logp.numel()>0,
            '新旧log概率需要同形状非空向量')
    require(new_logp.is_floating_point() and old_logp.is_floating_point()
            and new_logp.dtype==old_logp.dtype and new_logp.device==old_logp.device,
            '新旧log概率精度/设备非法')
    require(bool(torch.isfinite(new_logp).all() and torch.isfinite(old_logp).all()),'log概率非有限')
    delta=new_logp-old_logp.detach()
    require(bool((delta.detach().abs()<40).all()),'概率比过大，不静默截断')
    return delta.exp()


def policy_loss(new_logp, old_logp, advantage, epsilon):
    require(type(epsilon) in (int,float) and math.isfinite(epsilon) and 0<epsilon<1,'policy clip范围非法')
    require(isinstance(advantage,torch.Tensor) and advantage.shape==new_logp.shape
            and advantage.dtype==new_logp.dtype and advantage.device==new_logp.device
            and bool(torch.isfinite(advantage).all()),'优势形状/精度/数值非法')
    ratio=probability_ratio(new_logp,old_logp)
    a=advantage.detach()
    plain=ratio*a
    bounded=ratio.clamp(1-epsilon,1+epsilon)*a
    obj=torch.minimum(plain,bounded)
    outside=(ratio>1+epsilon)|(ratio<1-epsilon)
    active=((ratio>1+epsilon)&(a>0))|((ratio<1-epsilon)&(a<0))
    return -obj.mean(),{'ratio':ratio,'surrogate':obj,'plain':plain,
                        'ratio_outside_fraction':float(outside.detach().float().mean()),
                        'active_clip_fraction':float(active.detach().float().mean())}


def value_loss(prediction, old_value, target, epsilon=.2, method='clipmax'):
    """三种可选值损失；clipmax不是原始PPO论文的必选部件。

    plain:        0.5*MSE(pred,target)
    clipmax:      0.5*mean(max(raw_squared, clipped_pred_squared)) (PPO2变体)
    clipped_pred: 0.5*MSE(clipped_prediction,target) (SB3可选实现)
    """
    require(method in {'plain','clipmax','clipped_pred'},'value loss方法不支持')
    require(type(epsilon) in (float,int) and math.isfinite(epsilon) and epsilon>0,'value epsilon无效')
    require(all(isinstance(x,torch.Tensor) for x in (prediction,old_value,target))
            and prediction.ndim==1 and prediction.numel()>0
            and prediction.shape==old_value.shape==target.shape,'value输入必须同形状一维向量')
    require(all(x.is_floating_point() and x.dtype==prediction.dtype and x.device==prediction.device
                and bool(torch.isfinite(x).all()) for x in (prediction,old_value,target)),
            'value设备、dtype或数值非法')
    old=old_value.detach(); goal=target.detach()
    raw_squared=(prediction-goal).square()
    clipped_prediction=old+(prediction-old).clamp(-epsilon,epsilon)
    clipped_squared=(clipped_prediction-goal).square()
    if method=='plain': squared=raw_squared
    elif method=='clipmax': squared=torch.maximum(raw_squared,clipped_squared)
    else: squared=clipped_squared
    return .5*squared.mean(), {
        'raw_half_mse':float((.5*raw_squared.detach()).mean()),
        'clipped_prediction_half_mse':float((.5*clipped_squared.detach()).mean()),
        'max_half_mse':float((.5*torch.maximum(raw_squared.detach(),clipped_squared.detach())).mean()),
        'value_outside_clip_fraction':float(((prediction.detach()-old).abs()>epsilon).float().mean()),
        'method':method}


def normalize_advantages(advantages):
    require(isinstance(advantages,torch.Tensor) and advantages.ndim==1 and advantages.numel()>1
            and advantages.is_floating_point() and bool(torch.isfinite(advantages).all()),
            '归一化优势需要超过一个有限浮点样本')
    a=advantages.detach()
    return (a-a.mean())/(a.std(unbiased=False)+1e-8)


def old_state_kl(old_distribution,new_distribution):
    """仅计算旧样本访问的三状态上的Categorical KL(old||new)的平均。"""
    require(isinstance(old_distribution,torch.Tensor) and isinstance(new_distribution,torch.Tensor)
            and old_distribution.ndim==2 and old_distribution.shape==new_distribution.shape
            and old_distribution.shape[1]==2 and old_distribution.numel()>0,'KL分布形状无效')
    require(all(torch.isfinite(x).all() for x in (old_distribution,new_distribution)),'非有限分布')
    for d in (old_distribution,new_distribution):
        require(bool(torch.allclose(d.logsumexp(-1),torch.zeros_like(d[:,0]),atol=1e-5)),
                'KL需要归一化log概率')
    old=old_distribution.detach()
    return (old.exp()*(old-new_distribution)).sum(-1).mean()


def joint_objective(new_logps, old_logps, advantages, policy_log_distribution,
                    predicted_values, old_values, fixed_returns, cfg, value_method):
    cfg.validate()
    actor_raw,pi_info=policy_loss(new_logps,old_logps,advantages,cfg.clip_ratio)
    require(policy_log_distribution.shape==(new_logps.numel(),2), 'entropy分布shape非法')
    require(bool(torch.isfinite(policy_log_distribution).all()),'entropy分布非法')
    require(torch.allclose(policy_log_distribution.logsumexp(-1),
            torch.zeros_like(new_logps),atol=1e-5),'entropy需要归一化的log概率')
    entropy=-(policy_log_distribution.exp()*policy_log_distribution).sum(-1).mean()
    critic_raw,v_info=value_loss(predicted_values,old_values,fixed_returns,
                                cfg.clip_value,value_method)
    actor_total=actor_raw-cfg.entropy_coef*entropy
    critic_total=cfg.value_weight*critic_raw
    return actor_total,critic_total,{
        'policy_surrogate_loss':float(actor_raw.detach()),
        'entropy':float(entropy.detach()),
        'actor_loss':float(actor_total.detach()),
        'critic_raw_loss':float(critic_raw.detach()),
        'critic_weighted_loss':float(critic_total.detach()),
        'total_joint_loss':float((actor_total+critic_total).detach()),
        **{key:value for key,value in pi_info.items() if not isinstance(value,torch.Tensor)},
        **v_info}


def value_clipping_examples():
    """把value裁剪位置、损失选择、奖励量纲依赖拆开。"""
    rows=[]
    for label,pred,target in [('moved_toward_distant_target',.5,2.),
                              ('moved_away_from_target',.5,0.),
                              ('within_clip',.1,2.)]:
        for mode in ['plain','clipmax','clipped_pred']:
            x=torch.tensor([pred],dtype=torch.float64,requires_grad=True)
            old=torch.tensor([0.],dtype=torch.float64,requires_grad=True)
            y=torch.tensor([target],dtype=torch.float64,requires_grad=True)
            loss,_=value_loss(x,old,y,.2,mode)
            gx,go,gy=torch.autograd.grad(loss,(x,old,y),allow_unused=True)
            rows.append({'case':label,'method':mode,'loss':float(loss.detach()),
                         'new_value_gradient':float(gx),
                         'old_value_gradient':None if go is None else float(go),
                         'target_gradient':None if gy is None else float(gy)})
    scaling=[]
    for multiplier in [1.,10.]:
        x=torch.tensor([.5*multiplier],dtype=torch.float64,requires_grad=True)
        old=torch.zeros_like(x)
        target=torch.tensor([2.*multiplier],dtype=torch.float64)
        unscaled_loss,_=value_loss(x,old,target,.2,'clipmax')
        scaled_loss,_=value_loss(x,old,target,.2*multiplier,'clipmax')
        scaling.append({'reward_multiplier':multiplier,'epsilon_fixed':.2,
                        'loss_fixed_epsilon':float(unscaled_loss.detach()),
                        'loss_epsilon_scaled_with_reward':float(scaled_loss.detach()),
                        'normalized_loss_scaled_epsilon':float(scaled_loss.detach())/multiplier**2})
    return {'examples':rows,'scaling':scaling,
            'boundary':'三个值损失不是相同目标；clipmax/clip_pred均非硬约束，不保证改进critic'}

In [9]:
p_new=torch.tensor([math.log(.6),math.log(1.5),math.log(.6),math.log(1.5)],
                   dtype=torch.float64,requires_grad=True)
p_old=torch.zeros(4,dtype=torch.float64,requires_grad=True)
adv=torch.tensor([1.,1.,-1.,-1.],dtype=torch.float64,requires_grad=True)
loss,measure=policy_loss(p_new,p_old,adv,.2)
print('surrogate:',measure['surrogate'].detach().tolist())
print('clip激活占比:',measure['active_clip_fraction'])
grad=torch.autograd.grad(loss,(p_new,p_old,adv),allow_unused=True)
print('当前log概率梯度:',grad[0].tolist())
print('旧log概率梯度:',grad[1],'优势梯度:',grad[2])

surrogate: [0.6, 1.2, -0.8, -1.5]
clip激活占比: 0.5
当前log概率梯度: [-0.15, 0.0, 0.0, 0.375]
旧log概率梯度: None 优势梯度: None


## 5. Value clipping 有不止一种定义

本课**明确比较三种价值损失**。设当前预测 $v$、采样时旧价值 $v_{old}$、固定 λ-return 为 $R$，定义：

\[
v_{clip}=v_{old}+\operatorname{clip}(v-v_{old},-\epsilon_V,+\epsilon_V)
\]

| 名称 | 本课定义 | 说明 |
|---|---|---|
| `plain` | $\frac12(v-R)^2$ | 不裁剪，Spinning Up 的 Critic 默认路径 |
| `clipmax` | $\frac12\max((v-R)^2,(v_{clip}-R)^2)$ | PPO2 式保守 max 两路损失 |
| `clipped_pred` | $\frac12(v_{clip}-R)^2$ | 仅裁剪预测后 MSE；SB3 的可选 `clip_range_vf` 路径 |

**后两种不是同一个损失，且都不是原始 PPO 算法必选组件。** Stable-Baselines3 明确把价值裁剪描述为扩展选项，并提醒它依赖奖励尺度。[SB3 PPO](https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html) · [SB3 源码](https://github.com/DLR-RM/stable-baselines3/blob/master/stable_baselines3/ppo/ppo.py)

所有版本都必须使用**冻结的** `v_old` 和 `R`；它们不应收到 Critic 反向梯度。

In [10]:
value_demo=value_clipping_examples()
for case in ['moved_toward_distant_target','moved_away_from_target','within_clip']:
    print('\n',case)
    for row in value_demo['examples']:
        if row['case']==case:
            print(row['method'],'loss=',round(row['loss'],6),
                  'new_v_gradient=',round(row['new_value_gradient'],6))


 moved_toward_distant_target
plain loss= 1.125 new_v_gradient= -1.5
clipmax loss= 1.62 new_v_gradient= 0.0
clipped_pred loss= 1.62 new_v_gradient= 0.0

 moved_away_from_target
plain loss= 0.125 new_v_gradient= 0.5
clipmax loss= 0.125 new_v_gradient= 0.5
clipped_pred loss= 0.02 new_v_gradient= 0.0

 within_clip
plain loss= 1.805 new_v_gradient= -1.9
clipmax loss= 1.805 new_v_gradient= -1.9
clipped_pred loss= 1.805 new_v_gradient= -1.9


### 5.1 保守 `clipmax` 为什么有时不推动 Critic 继续靠近目标？

例：`old=0, target=2, current=0.5, epsilon=0.2`。未裁剪半 MSE 是 1.125，裁剪预测后的半 MSE 是 1.62。`clipmax` 选择1.62；当前预测已经越出clip区间，损失在局部进入平坦段，梯度为0。

反过来：`old=0,target=0,current=0.5`。这里新预测远离正确目标，`clipmax` 仍选择未裁剪的平方误差并保留回拉梯度；`clipped_pred` 却可在裁剪区间外给出平坦损失。

这能在单个样本上明确区分三种实现。全模型由于多样本共享参数、优化器动量等原因，**裁剪不会使当前价值估计被硬限制在旧值±epsilon内**。

### 5.2 奖励尺度为什么影响 value clip？

如果奖励与目标值从“元”换到“分”，乘以100，固定 `epsilon_V=0.2` 与相应缩放epsilon不是同一个优化问题。不能照搬一个数值而不核查奖励尺度。讲义的可运行实验会比较同时放大奖励与裁剪幅度后的归一化平方损失。

In [11]:
for row in value_demo['scaling']:
    print(row)
print('注意：两种clip模式的数学定义已在输出与脚本中明确区分。')

{'reward_multiplier': 1.0, 'epsilon_fixed': 0.2, 'loss_fixed_epsilon': 1.62, 'loss_epsilon_scaled_with_reward': 1.62, 'normalized_loss_scaled_epsilon': 1.62}
{'reward_multiplier': 10.0, 'epsilon_fixed': 0.2, 'loss_fixed_epsilon': 196.02, 'loss_epsilon_scaled_with_reward': 162.0, 'normalized_loss_scaled_epsilon': 1.62}
注意：两种clip模式的数学定义已在输出与脚本中明确区分。


## 6. 收集后冻结 Buffer，不能每轮重新生成目标

每批在 Actor 下采样真实回合，保存：

```text
states, actions, old_logps, old_action_distribution
old_values, raw_advantages, normalized_advantages, fixed_lambda_returns
terminated, truncated, complete_episodes
old_actor_hash, old_critic_hash, batch_id, max_epochs
```

本课对原始优势先做一次均值与总体标准差归一化，**仅用于 Actor**；Critic 始终回归没有标准化的 λ-return。零方差时中心化优势为0，不能伪造非零学习信号。

采样数据封存后，用同一批进行四轮更新；不能刷新 `old_logp`、旧 Critic 值、优势或返回目标，否则概率比与价值裁剪失去统一的行为快照。

哈希与数据封存仅用于检测课程实现被意外修改，不是针对恶意修改的安全签名。

In [12]:
def buffer_seal(buffer):
    keys=['states','actions','old_logps','old_distribution','old_values','fixed_returns',
          'raw_advantages','normalized_advantages','terminated','truncated']
    tensors={k:buffer[k] for k in keys}
    extra={'episodes':buffer['episodes'],'rollout_id':buffer['rollout_id'],
           'behavior_actor_hash':buffer['behavior_actor_hash'],
           'behavior_critic_hash':buffer['behavior_critic_hash'],
           'horizon':buffer['horizon'],'max_epochs':buffer['max_epochs']}
    return sha256({'tensors':tensor_hash(tensors),'metadata':extra})


def prepare_buffer(actor,critic, episodes, cfg, rollout_id):
    cfg.validate()
    require(type(rollout_id) is int and rollout_id>=0,'rollout编号非法')
    require(len(episodes)==cfg.episodes_per_update,'一次rollout应使用约定回合数')
    for ep in episodes: validate_episode(ep,cfg.horizon)
    targets=compute_targets(episodes,critic,cfg.gamma,cfg.gae_lambda,True)
    states=torch.tensor(targets['states'],dtype=torch.long)
    actions=torch.tensor(targets['actions'],dtype=torch.long)
    with torch.no_grad():
        actor.eval();critic.eval()
        distribution=F.log_softmax(actor(states),dim=-1).detach().clone()
        values=critic(states).detach().clone()
        old_lp=distribution.gather(-1,actions[:,None]).squeeze(-1).detach().clone()
    collected=torch.tensor(targets['old_logps'],dtype=torch.float32)
    require(torch.allclose(old_lp,collected,rtol=1e-4,atol=2e-5),
            '采样和重新计算的行为概率不同；策略已变化')
    old_v=torch.tensor(targets['values_old'],dtype=torch.float32)
    require(torch.allclose(values,old_v,rtol=1e-5,atol=1e-5),'原critic值和buffer不一致')
    raw=torch.tensor(targets['advantages'],dtype=torch.float32)
    normalized=normalize_advantages(raw)
    fixed_returns=torch.tensor(targets['lambda_returns'],dtype=torch.float32)
    require(torch.allclose(fixed_returns,raw+old_v,atol=2e-5,rtol=2e-5),
            'lambda-return必须使用同一份旧值与GAE')
    old_actor_hash=tensor_hash(actor.state_dict());old_critic_hash=tensor_hash(critic.state_dict())
    buffer={
        'states':states.clone(),'actions':actions.clone(),
        'old_logps':old_lp.clone(),'old_distribution':distribution.clone(),
        'old_values':old_v.clone(),'fixed_returns':fixed_returns.clone(),
        'raw_advantages':raw.clone(),'normalized_advantages':normalized.clone(),
        'terminated':torch.tensor(targets['terminated'],dtype=torch.bool),
        'truncated':torch.tensor(targets['truncated'],dtype=torch.bool),
        'episodes':[[asdict(step) for step in ep] for ep in episodes],
        'rollout_id':rollout_id,'horizon':cfg.horizon,'max_epochs':cfg.update_epochs,
        'behavior_actor_hash':old_actor_hash,'behavior_critic_hash':old_critic_hash,
        'expected_actor_hash':old_actor_hash,'expected_critic_hash':old_critic_hash,
        'actor_updates':0,'critic_updates':0,'actor_stopped':False,'closed':False,
        'sampled_episodes':len(episodes),'sampled_steps':len(states),
    }
    for key in ('states','actions','old_logps','old_distribution','old_values',
                'fixed_returns','raw_advantages','normalized_advantages','terminated','truncated'):
        require(not buffer[key].requires_grad,'buffer不应持有梯度图')
    buffer['seal']=buffer_seal(buffer)
    return buffer


def validate_buffer(actor,critic,buffer,cfg,rollout_id):
    require(not buffer['closed'],'这批数据已关闭；需要重新采样')
    require(type(rollout_id) is int and buffer['rollout_id']==rollout_id,'跨rollout混用')
    require(buffer['max_epochs']==cfg.update_epochs,'复用预算与训练约定不符')
    require(buffer['critic_updates']<cfg.update_epochs,'超过同批Critic最大更新次数')
    require(buffer['actor_updates']<=buffer['critic_updates'],'actor更新不可能多于critic')
    require(buffer['seal']==buffer_seal(buffer),'旧概率、旧价值、固定目标或rollout记录被修改')
    require(buffer['expected_actor_hash']==tensor_hash(actor.state_dict()),'actor发生控制循环外的更新')
    require(buffer['expected_critic_hash']==tensor_hash(critic.state_dict()),'critic发生控制循环外的更新')
    require(buffer['sampled_episodes']==cfg.episodes_per_update,'样本来源数量不符')
    require(all(not t.requires_grad for k,t in buffer.items() if isinstance(t,torch.Tensor)),
            'buffer不得包含参与训练的计算图')


@torch.no_grad()
def buffer_diagnostics(actor,critic,buffer,cfg):
    actor.eval();critic.eval()
    states,actions=buffer['states'],buffer['actions']
    distribution=F.log_softmax(actor(states),dim=-1)
    new_lp=distribution.gather(-1,actions[:,None]).squeeze(-1)
    ratio=probability_ratio(new_lp,buffer['old_logps'])
    a=buffer['normalized_advantages']
    v=critic(states)
    kl=old_state_kl(buffer['old_distribution'],distribution)
    active=((ratio>1+cfg.clip_ratio)&(a>0)) | ((ratio<1-cfg.clip_ratio)&(a<0))
    return {'sampled_state_kl_old_to_current':max(0.,float(kl)),
            'sampled_log_ratio_mean':float((new_lp-buffer['old_logps']).mean()),
            'ratio_min':float(ratio.min()),'ratio_max':float(ratio.max()),
            'ratio_outside_fraction':float(((ratio<1-cfg.clip_ratio)|(ratio>1+cfg.clip_ratio)).float().mean()),
            'active_clip_fraction':float(active.float().mean()),
            'mean_old_value':float(buffer['old_values'].mean()),
            'mean_current_value':float(v.mean()),
            'mean_lambda_return':float(buffer['fixed_returns'].mean()),
            'mean_raw_advantage':float(buffer['raw_advantages'].mean()),
            'normalized_advantage_std':float(a.std(unbiased=False)),
            'scope':'exact_action_KL_on_states_seen_in_old_rollout_not_global_state_distribution'}


def update_epoch(actor,critic,buffer,actor_opt,critic_opt,cfg,value_method,rollout_id):
    cfg.validate();validate_buffer(actor,critic,buffer,cfg,rollout_id)
    require(value_method in {'plain','clipmax','clipped_pred'},'未知value损失')
    actor_params=list(actor.parameters());critic_params=list(critic.parameters())
    a_opt={id(p) for group in actor_opt.param_groups for p in group['params']}
    c_opt={id(p) for group in critic_opt.param_groups for p in group['params']}
    require(a_opt=={id(p) for p in actor_params},'actor优化器参数错误')
    require(c_opt=={id(p) for p in critic_params},'critic优化器参数错误')
    require(not a_opt & c_opt,'actor与critic不得共享优化器参数对象')
    before=buffer_diagnostics(actor,critic,buffer,cfg)
    # KL停止是actor当前批次的门控。Critic仍按固定lambda targets继续回归。
    if not buffer['actor_stopped'] and before['sampled_state_kl_old_to_current']>cfg.target_state_kl:
        buffer['actor_stopped']=True
    update_actor=not buffer['actor_stopped']
    states,actions=buffer['states'],buffer['actions']
    actor.train();critic.train()
    distribution=F.log_softmax(actor(states),dim=-1)
    current_lp=distribution.gather(-1,actions[:,None]).squeeze(-1)
    values=critic(states)
    actor_loss,critic_loss,parts=joint_objective(
        current_lp,buffer['old_logps'],buffer['normalized_advantages'],distribution,
        values,buffer['old_values'],buffer['fixed_returns'],cfg,value_method)
    actor_opt.zero_grad(set_to_none=True);critic_opt.zero_grad(set_to_none=True)
    total=(actor_loss if update_actor else actor_loss.detach())+critic_loss
    total.backward()
    for p in critic_params:
        require(p.grad is not None and bool(torch.isfinite(p.grad).all()),'critic梯度异常')
    for p in actor_params:
        if update_actor:
            require(p.grad is not None and bool(torch.isfinite(p.grad).all()),'actor梯度异常')
        else:
            require(p.grad is None,'已KL停止的actor不应收到梯度')
    if update_actor:
        actor_norm=nn.utils.clip_grad_norm_(actor_params,cfg.grad_clip,error_if_nonfinite=True)
        actor_opt.step()
        buffer['actor_updates']+=1
    else: actor_norm=None
    critic_norm=nn.utils.clip_grad_norm_(critic_params,cfg.grad_clip,error_if_nonfinite=True)
    critic_opt.step()
    buffer['critic_updates']+=1
    if buffer['critic_updates']>=buffer['max_epochs']:
        buffer['closed']=True
    buffer['expected_actor_hash']=tensor_hash(actor.state_dict())
    buffer['expected_critic_hash']=tensor_hash(critic.state_dict())
    require(buffer['seal']==buffer_seal(buffer),'更新不应修改冻结样本')
    after=buffer_diagnostics(actor,critic,buffer,cfg)
    return {'epoch':buffer['critic_updates'],'actor_updated':update_actor,
            'critic_updated':True,'actor_stopped':buffer['actor_stopped'],
            'before':before,'after':after,'loss_components':parts,
            'actor_grad_norm':None if actor_norm is None else float(actor_norm),
            'critic_grad_norm':float(critic_norm),
            'fixed_steps':buffer['sampled_steps']}

In [13]:
torch.set_num_threads(1)
seed_all(6262)
small_cfg=Config(episodes_per_update=8,update_epochs=3,
                 actor_hidden=16,critic_hidden=16)
actor_probe=Actor(16);critic_probe=Critic(16)
episodes_probe=collect_rollouts(actor_probe,8,small_cfg.horizon,random.Random(22))
buffer_probe=prepare_buffer(actor_probe,critic_probe,episodes_probe,small_cfg,0)
print('采样回合：',len(episodes_probe),'真实步骤：',buffer_probe['sampled_steps'])
print('采样时旧价值：',buffer_probe['old_values'][:4].tolist())
print('未标准化优势：',buffer_probe['raw_advantages'][:4].tolist())
print('λ-return：',buffer_probe['fixed_returns'][:4].tolist())
print('初始比值与KL：',buffer_diagnostics(actor_probe,critic_probe,buffer_probe,small_cfg)['ratio_max'],
      buffer_diagnostics(actor_probe,critic_probe,buffer_probe,small_cfg)['sampled_state_kl_old_to_current'])

采样回合： 8 真实步骤： 43
采样时旧价值： [0.5776156187057495, 0.5776156187057495, 0.587277889251709, 0.5613468885421753]
未标准化优势： [0.13858067989349365, 0.1966332048177719, 0.25078651309013367, 0.34879207611083984]
λ-return： [0.7161962985992432, 0.7742488384246826, 0.838064432144165, 0.9101389646530151]
初始比值与KL： 1.0 0.0


## 7. 一次联合 backward，两组优化器各管各的参数

训练一次需要核对旧状态、计算当前 Actor 分布和 Critic 输出，然后：

```text
policy_surrogate = -mean(min(ratio*A, clipped_ratio*A))
actor_loss = policy_surrogate - entropy_coef * entropy
critic_loss = value_weight * value_loss(current_V, old_V, fixed_lambda_return)
loss = actor_loss + critic_loss
```

只有当前 Actor 参数、当前 Critic 参数可导。旧策略、旧价值、目标与优势都必须停止梯度。代码在反向后检查每个网络梯度非空且有限，再分别裁剪梯度、调用自己的优化器。

完整批次重复四轮是教学简化，没有随机 minibatch。完整 PPO 工程通常包含分批次、多 epoch 与更多停止规则，本课不冒充已涵盖所有优化细节。

In [14]:
optimizer_actor=torch.optim.Adam(actor_probe.parameters(),lr=small_cfg.actor_lr)
optimizer_critic=torch.optim.Adam(critic_probe.parameters(),lr=small_cfg.critic_lr)
old_buffer_hash=buffer_probe['seal']
old_critic_values=buffer_probe['old_values'].clone()
first_update=update_epoch(actor_probe,critic_probe,buffer_probe,
                          optimizer_actor,optimizer_critic,small_cfg,'clipmax',0)
print('Actor更新：',first_update['actor_updated'])
print('Critic更新：',first_update['critic_updated'])
print('本次损失：',first_update['loss_components'])
print('旧值未修改：',torch.equal(buffer_probe['old_values'],old_critic_values))
print('Buffer封存状态不变：',old_buffer_hash==buffer_probe['seal']==buffer_seal(buffer_probe))

Actor更新： True
Critic更新： True
本次损失： {'policy_surrogate_loss': 3.188155517364066e-08, 'entropy': 0.6675453782081604, 'actor_loss': -0.006675422191619873, 'critic_raw_loss': 0.027793487533926964, 'critic_weighted_loss': 0.013896743766963482, 'total_joint_loss': 0.007221321575343609, 'ratio_outside_fraction': 0.0, 'active_clip_fraction': 0.0, 'raw_half_mse': 0.027793487533926964, 'clipped_prediction_half_mse': 0.027793487533926964, 'max_half_mse': 0.027793487533926964, 'value_outside_clip_fraction': 0.0, 'method': 'clipmax'}
旧值未修改： True
Buffer封存状态不变： True


## 8. KL 停止 Actor，为什么 Critic 仍然可以继续？

本课采用的诊断是**旧批次实际访问状态上的精确二动作 KL**：

\[
\mathbb E_{s\sim\text{old batch}}
D_{KL}(\pi_{old}(\cdot|s)\|\pi_{current}(\cdot|s))
\]

它不是对所有可能状态的完整 KL，也不是固定参考模型的 KL。与上一课的行为分布比较保持相同方向。

在下一轮联合优化之前，若该值严格超过阈值，就停止当前 rollout 的剩余 Actor 更新；**本课的设计仍让 Critic 继续针对这批固定 λ-return 回归，直到其总 epoch 预算结束**。

停止不会回滚上一轮已经造成的KL超限，也不保证 Actor 永远不越界。此处 Actor 与 Critic 有独立参数，所以这种单独门控更容易审计。

In [15]:
seed_all(6270)
stop_cfg=Config(episodes_per_update=8,update_epochs=3,actor_hidden=16,
                critic_hidden=16,actor_lr=.07,target_state_kl=1e-12)
a_stop=Actor(16);v_stop=Critic(16)
e_stop=collect_rollouts(a_stop,8,stop_cfg.horizon,random.Random(104))
b_stop=prepare_buffer(a_stop,v_stop,e_stop,stop_cfg,0)
o_a=torch.optim.Adam(a_stop.parameters(),lr=stop_cfg.actor_lr)
o_v=torch.optim.Adam(v_stop.parameters(),lr=stop_cfg.critic_lr)
for step in range(3):
    result=update_epoch(a_stop,v_stop,b_stop,o_a,o_v,stop_cfg,'clipmax',0)
    print('epoch',step+1,'Actor更新',result['actor_updated'],
          'Critic更新',result['critic_updated'],
          '更新前状态KL',result['before']['sampled_state_kl_old_to_current'])

epoch 1 Actor更新 True Critic更新 True 更新前状态KL 0.0
epoch 2 Actor更新 False Critic更新 True 更新前状态KL 0.3658795654773712
epoch 3 Actor更新 False Critic更新 True 更新前状态KL 0.3658795654773712


## 9. 三条价值损失分支，比较同一 Actor PPO-Clip

默认比较：

| 分支 | Actor loss | Critic loss |
|---|---|---|
| `plain` | 同一个 PPO-Clip＋熵 | `0.5*MSE` |
| `clipmax` | 同一个 PPO-Clip＋熵 | 保守 max-squared clip |
| `clipped_pred` | 同一个 PPO-Clip＋熵 | 裁剪预测值后 MSE |

固定初始 Actor/Critic 权重、问题采样种子调度与每批重复更新次数。各分支更新后 Actor 不同，即使采样种子相同，**实际动作轨迹仍可能不同**；不能称为完全相同样本实验。

默认 45 批×24 回合，三支各最多 180 次 Actor 和 180 次 Critic 更新。测试用**整个有限状态系统的精确成功概率**与每状态精确 Bellman 值的 RMSE，同时保留有限采样成功次数。

Critic 的目标是当前继续任务的价值；外部六步成功率是另一个指标。值预测降低误差，不证明 PPO 一定找到全局最优策略。

In [16]:
def train_arm(cfg, mode, schedule, base_actor, base_critic):
    require(mode in {'plain','clipmax','clipped_pred'},'不支持的模式')
    actor=copy.deepcopy(base_actor);critic=copy.deepcopy(base_critic)
    actor_opt=torch.optim.Adam(actor.parameters(),lr=cfg.actor_lr)
    critic_opt=torch.optim.Adam(critic.parameters(),lr=cfg.critic_lr)
    history=[];total_actor=total_critic=0;stopped=0;observed_steps=0
    for rollout_id,seed in enumerate(schedule):
        episodes=collect_rollouts(actor,cfg.episodes_per_update,cfg.horizon,random.Random(seed))
        buffer=prepare_buffer(actor,critic,episodes,cfg,rollout_id)
        first=buffer_diagnostics(actor,critic,buffer,cfg)
        require(abs(first['ratio_max']-1)<2e-5 and abs(first['ratio_min']-1)<2e-5,
                '新rollout的初始行为比不为1')
        row={'rollout_id':rollout_id,'episodes':len(episodes),
             'steps':buffer['sampled_steps'],
             'terminations':sum(ep[-1].terminated for ep in episodes),
             'truncations':sum(ep[-1].truncated for ep in episodes),
             'first':first,'epochs':[]}
        for _ in range(cfg.update_epochs):
            update=update_epoch(actor,critic,buffer,actor_opt,critic_opt,cfg,mode,rollout_id)
            row['epochs'].append(update)
            total_actor+=int(update['actor_updated']);total_critic+=int(update['critic_updated'])
        require(buffer['closed'],'复用达到上限应关闭')
        stopped+=int(buffer['actor_stopped']);observed_steps+=buffer['sampled_steps']
        if rollout_id==0 or (rollout_id+1)%15==0 or rollout_id+1==cfg.updates:
            ev=evaluate_policy(actor,critic,cfg.gamma,cfg.horizon,60,seed=6206)
            row['dev_observation']={'success_within_horizon':ev['exact_success_within_limit'],
                                    'value_rmse':ev['value_rmse']}
        history.append(row)
    return actor,critic,{
        'mode':mode,'history':history,'rollouts':cfg.updates,
        'actor_updates':total_actor,'critic_updates':total_critic,
        'actor_stopped_batches':stopped,'sampled_episodes':cfg.updates*cfg.episodes_per_update,
        'sampled_steps':observed_steps,'selection':'fixed_last_step_no_test_tuning'}


def save_json(path,value):
    Path(path).write_text(json.dumps(value,ensure_ascii=False,sort_keys=True,
                                    indent=2,allow_nan=False)+'\n',encoding='utf-8')


def experiment(cfg=None,root='lesson_62_outputs',mode='all'):
    cfg=cfg or Config();cfg.validate()
    require(mode in {'all','plain','clipmax','clipped_pred'},'mode非法')
    torch.set_num_threads(1);seed_all(cfg.seed)
    init_actor=Actor(cfg.actor_hidden)
    init_critic=Critic(cfg.critic_hidden)
    initial_a=tensor_hash(init_actor.state_dict());initial_c=tensor_hash(init_critic.state_dict())
    rng=random.Random(cfg.seed+1)
    schedule=[rng.randrange(2**31) for _ in range(cfg.updates)]
    modes=['plain','clipmax','clipped_pred'] if mode=='all' else [mode]
    root=Path(root);root.mkdir(parents=True,exist_ok=True)
    folder=root/('run_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
    folder.mkdir(exist_ok=False)
    manifest={'version':VERSION,'status':'running','python':platform.python_version(),
              'torch':str(torch.__version__),'device':'cpu','config':asdict(cfg),'sources':SOURCES,
              'limitations':['synthetic_3_state_continuing_MDP_external_horizon',
                             'actor_PPO_clip_GAE_and_optional_value_clipping_no_minibatch',
                             'independent_actor_critic_not_pretrained_LLM',
                             'value_clip_variant_specific_reward_scale_sensitive',
                             'no_exact_resume_state_single_seed_no_algorithm_ranking']}
    save_json(folder/'manifest.json',manifest)
    try:
        initial=evaluate_policy(init_actor,init_critic,cfg.gamma,cfg.horizon,cfg.eval_episodes)
        report={'baseline':initial,'actor_start_hash':initial_a,'critic_start_hash':initial_c,
                'schedule_sha256':sha256(schedule),'arms':{},'selection':'fixed_last_update'}
        for name in modes:
            print('Actor/Critic PPO arm',name,flush=True)
            actor,critic,training=train_arm(cfg,name,schedule,init_actor,init_critic)
            final=evaluate_policy(actor,critic,cfg.gamma,cfg.horizon,cfg.eval_episodes)
            report['arms'][name]={**training,'final':final,
                'start_actor_unmodified':tensor_hash(init_actor.state_dict())==initial_a,
                'start_critic_unmodified':tensor_hash(init_critic.state_dict())==initial_c}
            payload={'version':VERSION,'arm':name,'config':asdict(cfg),
                     'actor':{k:v.detach().cpu().clone() for k,v in actor.state_dict().items()},
                     'critic':{k:v.detach().cpu().clone() for k,v in critic.state_dict().items()},
                     'actor_sha256':tensor_hash(actor.state_dict()),
                     'critic_sha256':tensor_hash(critic.state_dict())}
            torch.save(payload,folder/(name+'.pt'))
        save_json(folder/'training.json',report)
        save_json(folder/'value_clipping_math.json',value_clipping_examples())
        save_json(folder/'gae_math.json',{
            'source':'lesson61_formula_recomputed_locally',
            'truncation_bias':exact_time_limit_bias(gamma=cfg.gamma,horizon=cfg.horizon),
            'contract':'only true terminal sets V(next)=0; truncation bootstraps but stops GAE trace'})
        save_json(folder/'reload.json',reload_run(folder))
        manifest['status']='completed';save_json(folder/'manifest.json',manifest)
        print('完成：',folder.resolve(),flush=True)
        return folder
    except Exception as exc:
        manifest.update(status='failed',error=type(exc).__name__+': '+str(exc))
        save_json(folder/'manifest.json',manifest)
        raise


def reload_run(folder):
    folder=Path(folder);require(folder.is_dir(),'运行目录不存在')
    stored=json.loads((folder/'training.json').read_text(encoding='utf-8'))
    outputs={}
    for name,item in sorted(stored['arms'].items()):
        ck=torch.load(folder/(name+'.pt'),map_location='cpu',weights_only=True)
        require(ck['version']==VERSION and ck['arm']==name,'检查点来源错误')
        cfg=Config(**ck['config']);cfg.validate()
        require(tensor_hash(ck['actor'])==ck['actor_sha256'] and
                tensor_hash(ck['critic'])==ck['critic_sha256'],'检查点权重摘要不符')
        actor=Actor(cfg.actor_hidden).requires_grad_(False).eval()
        critic=Critic(cfg.critic_hidden).requires_grad_(False).eval()
        actor.load_state_dict(ck['actor'],strict=True)
        critic.load_state_dict(ck['critic'],strict=True)
        result=evaluate_policy(actor,critic,cfg.gamma,cfg.horizon,cfg.eval_episodes)
        previous=item['final']
        err=max(abs(result[k]-previous[k]) for k in ('exact_success_within_limit','value_rmse'))
        require(err<1e-9 and result['forward_probabilities']==previous['forward_probabilities'],
                '重载指标发生变化')
        outputs[name]={'max_abs_error':err,
                       'trainable_parameters':sum(p.numel() for p in
                        list(actor.parameters())+list(critic.parameters()) if p.requires_grad)}
    return {'status':'passed','arms':outputs,'scope':'trusted_inference_reload_not_exact_training_resume'}

In [17]:
RUN_FULL_EXPERIMENT=True
OUTPUT_ROOT=Path('lesson_62_outputs')
RESULT_DIR=None
if RUN_FULL_EXPERIMENT:
    RESULT_DIR=experiment(Config(),OUTPUT_ROOT,'all')
else:
    print('已跳过完整训练，没有伪造结果。')

Actor/Critic PPO arm plain


Actor/Critic PPO arm clipmax


Actor/Critic PPO arm clipped_pred


完成： /mnt/data/_lesson62_work/notebook_run/lesson_62_outputs/run_20261103T080629_861850Z


In [18]:
if RESULT_DIR is not None:
    trained=json.loads((RESULT_DIR/'training.json').read_text(encoding='utf-8'))
    print('方法           6步精确成功率   价值RMSE       Actor/Critic更新   KL停止批次')
    print(f"{'初始策略':14}{trained['baseline']['exact_success_within_limit']:.6f}      "
          f"{trained['baseline']['value_rmse']:.7f}           --")
    for name,row in trained['arms'].items():
        print(f"{name:14}{row['final']['exact_success_within_limit']:.6f}      "
              f"{row['final']['value_rmse']:.7f}        "
              f"{row['actor_updates']}/{row['critic_updates']}             {row['actor_stopped_batches']}")

方法           6步精确成功率   价值RMSE       Actor/Critic更新   KL停止批次
初始策略          0.698056      1.0010730           --
clipmax       1.000000      0.0000332        172/180             3
clipped_pred  1.000000      0.0007268        174/180             2
plain         1.000000      0.0001710        173/180             3


### 如何阅读实际结果？

这次 CPU 固定配置中，三组都学到了几乎必定前进的策略，因此六步精确成功率接近1，但 Critic 的最终 RMSE 并不完全一样。`clipmax`、`plain` 和 `clipped_pred` 也不必在所有数据与随机种子下保持这个顺序。

**不要把选择较低价值 RMSE 就等于真实任务更好，或把更多 Critic 更新理解成 Actor 的训练更多。** Actor 可能因 KL 停止而少更新，Critic 仍可完成全部预定次数。

本课程的单种子有限任务，只验证工程路径与反例，不能用作 PPO 变体通用排名。

## 10. 文件身份与重载

完整运行建立独立时间戳目录，包括：

| 文件 | 内容 |
|---|---|
| `plain.pt`、`clipmax.pt`、`clipped_pred.pt` | Actor/Critic 权重、配置、摘要 |
| `training.json` | 所有 rollout 与 epoch 更新记录、Actor停止、Critic损失 |
| `value_clipping_math.json` | 三种损失的梯度与尺度反例 |
| `gae_math.json` | 精确截断自举偏差 |
| `reload.json` | 独立对象重载的数值一致性 |
| `checks.json`、`mutations.json` | 固定检查与故意错误检测 |

保存文件用于**推理结果重建**，没有保存完整优化器、随机状态和数据游标，不是精确断点续训。仅加载自己制作或可信来源的文件。

In [19]:
if RESULT_DIR is not None:
    print('重载：',json.dumps(reload_run(RESULT_DIR),ensure_ascii=False,indent=2))

重载： {
  "status": "passed",
  "arms": {
    "clipmax": {
      "max_abs_error": 0.0,
      "trainable_parameters": 0
    },
    "clipped_pred": {
      "max_abs_error": 0.0,
      "trainable_parameters": 0
    },
    "plain": {
      "max_abs_error": 0.0,
      "trainable_parameters": 0
    }
  },
  "scope": "trusted_inference_reload_not_exact_training_resume"
}


## 11. 自动检查：不能只看 loss 会不会下降

本课检查不仅涉及单元公式，还实际构造神经网络并更新参数：

- 真终止与外部截断、四种 λ 情况、精确自举偏差。
- 正负优势 PPO 裁剪、KL、熵与梯度冻结。
- `clipmax` 与 `clipped_pred` 各自的梯度和目标。
- 旧策略、旧 Critic、λ-return、标准化优势及完整采样记录的封存。
- Actor/Critic 独立优化器、批次次数上限、KL 触发后只停止 Actor。

数学检查和小模型检查通过不代表形式证明、真实机器人安全认证或预训练语言模型效果。

In [20]:
class Checks:
    def __init__(self): self.names=[]
    def ok(self,name,condition):
        if not bool(condition): raise AssertionError(name)
        self.names.append(name)
    def close(self,name,got,want,atol=1e-9,rtol=1e-9):
        a=torch.as_tensor(got,dtype=torch.float64)
        b=torch.as_tensor(want,dtype=torch.float64)
        self.ok(name,torch.allclose(a,b,atol=atol,rtol=rtol))
    def raises(self,name,call):
        try: call()
        except (LessonError,ValueError,TypeError): self.names.append(name)
        else: raise AssertionError(name+': 本应拒绝')


def run_checks():
    torch.set_num_threads(1);seed_all(6205);c=Checks()
    cfg=Config();cfg.validate()
    c.ok('默认30秒以内CPU教学配置',cfg.updates==45 and cfg.update_epochs==4)
    for name,value in [('actor_lr',float('nan')),('clip_ratio',1.),('clip_value',0.),
                       ('target_state_kl',-1.),('entropy_coef',-1.)]:
        c.raises('配置拒绝'+name,lambda k=name,v=value:Config(**{k:v}).validate())
    c.raises('config拒绝bool步数',lambda:Config(updates=True).validate())
    c.raises('config拒绝GAEλ越界',lambda:Config(gae_lambda=2.).validate())
    c.raises('config拒绝复用过多',lambda:Config(update_epochs=9).validate())
    c.ok('时间外截断不是终止',transition(1,0,5,6)==(1,-.01,False,True))
    c.ok('触达目标才终止',transition(2,1,5,6)==(3,.99,True,False))
    ep=[Step(0,0,-.01,0,False,False,-.69),Step(0,0,-.01,0,False,True,-.69)]
    vs=[.6,.8];ns=[.8,.5];g=.9
    first=-.01+g*.8-.6;last=-.01+g*.5-.8
    for lam in [0.,.5,.95,1.]:
        res=generalized_advantage_estimation([ep],vs,ns,g,lam)
        c.close('GAE λ'+str(lam),res['advantage'],[first+g*lam*last,last])
        c.close('Return=旧V+GAE λ'+str(lam),res['lambda_target'],
                [vs[0]+first+g*lam*last,vs[1]+last])
    c.close('外部截断应bootstrap',last,-.01+g*.5-.8)
    full=exact_time_limit_bias(gamma=.95,horizon=6)
    c.ok('精确枚举外部截断具有系统偏差',full['lost_expected_value']>0)
    c.close('bootstrap无偏于继续价值',full['proper_bootstrap_expected_return'],
            full['true_continuing_value_start'],atol=1e-8)
    c.ok('截断不是读下一个回合开头',ep[-1].truncated and ep[-1].next_obs==0)

    pred=torch.tensor([.5],dtype=torch.float64,requires_grad=True)
    old=torch.tensor([0.],dtype=torch.float64,requires_grad=True)
    goal=torch.tensor([2.],dtype=torch.float64,requires_grad=True)
    numbers={}
    for mode in ['plain','clipmax','clipped_pred']:
        loss,info=value_loss(pred,old,goal,.2,mode)
        gp,go,gr=torch.autograd.grad(loss,(pred,old,goal),allow_unused=True)
        numbers[mode]=(float(loss.detach()),float(gp),go,gr)
        c.ok(mode+'参考和固定目标不反传',go is None and gr is None)
    c.close('plain目标.5*(.5-2)^2',numbers['plain'][0],1.125)
    c.close('clipmax取较大损失',numbers['clipmax'][0],1.62)
    c.close('clipped_pred单独裁剪预测',numbers['clipped_pred'][0],1.62)
    c.close('plain朝目标梯度',numbers['plain'][1],-1.5)
    c.close('clipmax变平坦',numbers['clipmax'][1],0.)
    c.close('clipped_pred变平坦',numbers['clipped_pred'][1],0.)
    for mode in ['plain','clipmax','clipped_pred']:
        x=torch.tensor([.5],dtype=torch.float64,requires_grad=True)
        zero=torch.tensor([0.],dtype=torch.float64)
        loss,_=value_loss(x,zero,zero,.2,mode)
        gradient=torch.autograd.grad(loss,x)[0]
        if mode=='clipped_pred':
            c.close('clipped_pred远离目标也可能平坦',gradient,[0.])
            c.close('clipped_pred远离目标loss较小',loss,.02)
        else:
            c.close(mode+'远离目标应有恢复梯度',gradient,[.5])
            c.close(mode+'远离目标loss',loss,.125)
    for invalid in [0.,float('nan'),-1.,True]:
        c.raises('value epsilon非法'+str(invalid),lambda z=invalid:value_loss(pred,old,goal,z))
    c.raises('value形状不符',lambda:value_loss(pred,torch.zeros(2),goal))
    c.raises('value方法未知',lambda:value_loss(pred,old,goal,.2,'not_a_method'))
    vexpr=value_clipping_examples()
    c.ok('值损失方法齐全',len(vexpr['examples'])==9)
    c.close('奖励尺度同时放大epsilon使归一化loss保持',
        [r['normalized_loss_scaled_epsilon'] for r in vexpr['scaling']], [1.62,1.62],atol=1e-7)
    c.ok('不随奖励单位调整epsilon会改变相对clip尺度',
         vexpr['scaling'][0]['loss_fixed_epsilon']!=vexpr['scaling'][1]['loss_fixed_epsilon']/100.)
    a=torch.tensor([1.,-1.,1.,-1.],dtype=torch.float64,requires_grad=True)
    norm=normalize_advantages(a)
    c.close('标准化优势总体均值为0',float(norm.mean()),0.)
    c.close('标准化优势总体std近似1',float(norm.std(unbiased=False)),1.,atol=1e-8)
    c.ok('优势不参与梯度',not norm.requires_grad)
    c.close('等值优势标准化为0',normalize_advantages(torch.ones(4)),torch.zeros(4))
    c.raises('只有一条不能归一化',lambda:normalize_advantages(torch.ones(1)))
    oldlp=torch.zeros(4,dtype=torch.float64,requires_grad=True)
    current=torch.tensor([math.log(.6),math.log(1.5),math.log(.6),math.log(1.5)],
                         dtype=torch.float64,requires_grad=True)
    adv=torch.tensor([1.,1.,-1.,-1.],dtype=torch.float64,requires_grad=True)
    loss,info=policy_loss(current,oldlp,adv,.2)
    gc,go,ga=torch.autograd.grad(loss,(current,oldlp,adv),allow_unused=True)
    c.close('PPO正负优势四象限surrogate',info['surrogate'],[.6,1.2,-.8,-1.5])
    c.close('PPO四象限梯度',gc,[-.15,0.,0.,.375])
    c.ok('old策略没有梯度',go is None)
    c.ok('优势没有梯度',ga is None)
    c.close('PPO超界比例',info['ratio_outside_fraction'],1.)
    c.close('PPO真正有效裁剪比例',info['active_clip_fraction'],.5)
    c.raises('PPO epsilon非法',lambda:policy_loss(current,oldlp,adv,1.))
    c.raises('PPO比值过大拒绝',lambda:probability_ratio(torch.tensor([100.]),torch.zeros(1)))
    c.raises('PPO优势含NaN拒绝',lambda:policy_loss(current,oldlp,torch.full_like(adv,float('nan')),.2))
    same=torch.tensor([-1.,-2.],dtype=torch.float64,requires_grad=True)
    ratio=probability_ratio(same,same.detach().clone())
    c.close('新rollout起点ratio为1',ratio,[1.,1.])
    c.close('新旧同值不代表梯度为0',torch.autograd.grad(ratio.sum(),same)[0],[1.,1.])
    log_old=torch.tensor([[-math.log(2),-math.log(2)]],dtype=torch.float64)
    c.close('同策略KL为0',old_state_kl(log_old,log_old),0.)
    sample_new=torch.tensor([[-math.log(4),math.log(.75)]],dtype=torch.float64)
    c.close('旧策略到新策略KL手算',old_state_kl(log_old,sample_new),
            .5*math.log(.5/.25)+.5*math.log(.5/.75))
    c.raises('未归一化KL输入拒绝',lambda:old_state_kl(log_old+1,sample_new))

    seed_all(6231)
    actor=Actor(16);critic=Critic(16)
    cfg=Config(episodes_per_update=8,update_epochs=3,actor_hidden=16,critic_hidden=16)
    eps=collect_rollouts(actor,cfg.episodes_per_update,cfg.horizon,random.Random(100))
    buf=prepare_buffer(actor,critic,eps,cfg,0)
    c.ok('episode都可校验',all(validate_episode(ep,cfg.horizon) is None for ep in eps))
    c.ok('rollout数量8',len(eps)==8)
    c.ok('有训练步骤',buf['sampled_steps']>=8)
    c.ok('每个Buffer保存旧策略与旧价值',buf['old_distribution'].shape==(buf['sampled_steps'],2)
         and buf['old_values'].shape==(buf['sampled_steps'],))
    c.close('固定lambda return等于旧值+原始优势',buf['fixed_returns'],
            buf['old_values']+buf['raw_advantages'],atol=2e-5)
    c.ok('不会让buffer持有梯度',all(not t.requires_grad for t in buf.values() if isinstance(t,torch.Tensor)))
    initial=buffer_diagnostics(actor,critic,buf,cfg)
    c.close('新rollout全ratio为1',initial['ratio_max'],1.,atol=1e-5)
    c.close('新rollout采样状态KL为0',initial['sampled_state_kl_old_to_current'],0.,atol=1e-7)
    c.ok('actor与critic参数没有共享存储',not {p.data_ptr() for p in actor.parameters()} &
         {p.data_ptr() for p in critic.parameters()})
    c.ok('旧策略摘要与当前相同',buf['behavior_actor_hash']==tensor_hash(actor.state_dict()))
    c.ok('旧critic摘要与当前相同',buf['behavior_critic_hash']==tensor_hash(critic.state_dict()))
    # 把截断实例的真V(next)与错处理对应起来：不得意外回归终止=0的目标。
    extra=[Step(0,0,-.01,0,False,False,-.69),
           Step(0,0,-.01,0,False,False,-.69),
           Step(0,0,-.01,0,False,True,-.69)]
    corr=compute_targets([extra],critic,.95,.95,True)
    wrong=compute_targets([extra],critic,.95,.95,False)
    c.close('正确截断与错误处理的TD差值',
            corr['deltas'][-1]-wrong['deltas'][-1], .95*corr['next_values_old'][-1])
    c.ok('截断仍保存非终止next state',extra[-1].truncated and not extra[-1].terminated)
    bad=copy.deepcopy(buf);bad['old_logps'][0]+=.1
    c.raises('旧策略分母被改拒绝',lambda:validate_buffer(actor,critic,bad,cfg,0))
    bad=copy.deepcopy(buf);bad['old_values'][0]+=1
    c.raises('旧Value快照被改拒绝',lambda:validate_buffer(actor,critic,bad,cfg,0))
    bad=copy.deepcopy(buf);bad['fixed_returns'][0]+=1
    c.raises('固定lambda return被改拒绝',lambda:validate_buffer(actor,critic,bad,cfg,0))
    bad=copy.deepcopy(buf);bad['normalized_advantages'][0]+=1
    c.raises('固定优势被改拒绝',lambda:validate_buffer(actor,critic,bad,cfg,0))
    bad=copy.deepcopy(buf);bad['episodes'][0][0]['reward']+=1
    c.raises('采样记录被改拒绝',lambda:validate_buffer(actor,critic,bad,cfg,0))
    c.raises('跨rollout混用拒绝',lambda:validate_buffer(actor,critic,buf,cfg,1))
    aopt=torch.optim.Adam(actor.parameters(),lr=cfg.actor_lr)
    vopt=torch.optim.Adam(critic.parameters(),lr=cfg.critic_lr)
    badopt=torch.optim.Adam(list(actor.parameters())+list(critic.parameters()),lr=.01)
    c.raises('Actor优化器包含Critic拒绝',lambda:update_epoch(actor,critic,buf,badopt,vopt,cfg,'plain',0))
    badopt2=torch.optim.Adam(list(actor.parameters()),lr=.01)
    c.raises('Critic优化器参数错配拒绝',lambda:update_epoch(actor,critic,buf,aopt,badopt2,cfg,'plain',0))
    orig_seal=buf['seal']; orig_old=buf['old_values'].clone();orig_return=buf['fixed_returns'].clone()
    a_before=tensor_hash(actor.state_dict());v_before=tensor_hash(critic.state_dict())
    first=update_epoch(actor,critic,buf,aopt,vopt,cfg,'plain',0)
    c.ok('真实联合梯度第一轮成功',first['actor_updated'] and first['critic_updated'])
    c.ok('Actor参数确实更新',tensor_hash(actor.state_dict())!=a_before)
    c.ok('Critic参数确实更新',tensor_hash(critic.state_dict())!=v_before)
    c.ok('Actor与Critic各有有限梯度范数',math.isfinite(first['actor_grad_norm']) and math.isfinite(first['critic_grad_norm']))
    c.close('旧价值快照仍固定',buf['old_values'],orig_old,atol=0,rtol=0)
    c.close('旧lambda return仍固定',buf['fixed_returns'],orig_return,atol=0,rtol=0)
    c.ok('seal不变',buf['seal']==orig_seal==buffer_seal(buf))
    for _ in range(2):update_epoch(actor,critic,buf,aopt,vopt,cfg,'plain',0)
    c.ok('完整批次正好三轮critic更新',buf['critic_updates']==3)
    c.ok('达到复用上限批次关闭',buf['closed'])
    c.raises('第四轮重复训练被拒绝',lambda:update_epoch(actor,critic,buf,aopt,vopt,cfg,'plain',0))

    # 训练后重启一个rollout，Old/Current分别重新封存，一开始ratio恢复1。
    new_eps=collect_rollouts(actor,cfg.episodes_per_update,cfg.horizon,random.Random(101))
    new_buf=prepare_buffer(actor,critic,new_eps,cfg,1)
    c.close('下一个rollout开始ratio重新为1',buffer_diagnostics(actor,critic,new_buf,cfg)['ratio_max'],1.,atol=2e-5)
    c.ok('新旧行为hash不同',new_buf['behavior_actor_hash']!=buf['behavior_actor_hash'])
    c.ok('新旧critic hash不同',new_buf['behavior_critic_hash']!=buf['behavior_critic_hash'])
    # KL停止：先在极小阈值下进行一次Actor更新，再应冻结Actor但继续训练Critic。
    cfg_stop=Config(episodes_per_update=8,update_epochs=3,actor_lr=.07,critic_lr=.02,
                    target_state_kl=1e-12,actor_hidden=16,critic_hidden=16)
    a2=Actor(16);v2=Critic(16)
    ep2=collect_rollouts(a2,8,cfg_stop.horizon,random.Random(104))
    b2=prepare_buffer(a2,v2,ep2,cfg_stop,0)
    oa=torch.optim.Adam(a2.parameters(),lr=cfg_stop.actor_lr)
    ov=torch.optim.Adam(v2.parameters(),lr=cfg_stop.critic_lr)
    start=update_epoch(a2,v2,b2,oa,ov,cfg_stop,'clipmax',0)
    c.ok('KL停止实验第一步先更新',start['actor_updated'])
    before_stop=tensor_hash(a2.state_dict());v_before_stop=tensor_hash(v2.state_dict())
    second=update_epoch(a2,v2,b2,oa,ov,cfg_stop,'clipmax',0)
    c.ok('KL超阈值之后Actor停止',not second['actor_updated'] and second['actor_stopped'])
    c.ok('已停止Actor保持不变',tensor_hash(a2.state_dict())==before_stop)
    c.ok('已停止Actor仍允许Critic更新',tensor_hash(v2.state_dict())!=v_before_stop)
    c.ok('Actor停止后Critic继续是明确设计',second['critic_updated'])
    third=update_epoch(a2,v2,b2,oa,ov,cfg_stop,'clipmax',0)
    c.ok('本批Actor不会重新启动',not third['actor_updated'])
    c.ok('Critic仍完成三次epoch',b2['critic_updates']==3)
    c.ok('KL超过阈值不表示自动回滚',second['before']['sampled_state_kl_old_to_current']>cfg_stop.target_state_kl)
    c.ok('Actor已停止时梯度必须None',all(p.grad is None for p in a2.parameters()))
    c.raises('已停止批次继续调用拒绝',lambda:update_epoch(a2,v2,b2,oa,ov,cfg_stop,'clipmax',0))

    # 联合目标必须只让自己的参数获得梯度，不能让旧critic或优势反向。
    x=torch.tensor([.1,.3],dtype=torch.float64,requires_grad=True)
    oldlp=torch.tensor([0.,0.],dtype=torch.float64,requires_grad=True)
    advantage=torch.tensor([1.,-1.],dtype=torch.float64,requires_grad=True)
    actor_logs=F.log_softmax(torch.stack([x,-x],-1),-1)
    vn=torch.tensor([.5,0.],dtype=torch.float64,requires_grad=True)
    vo=torch.tensor([0.,0.],dtype=torch.float64,requires_grad=True)
    ret=torch.tensor([2.,0.],dtype=torch.float64,requires_grad=True)
    cfg_math=Config(entropy_coef=.01)
    a_loss,v_loss,parts=joint_objective(actor_logs[:,0],oldlp,advantage,actor_logs,
                                        vn,vo,ret,cfg_math,'clipmax')
    grads=torch.autograd.grad(a_loss+v_loss,(x,oldlp,advantage,vn,vo,ret),allow_unused=True)
    c.ok('联合Actor获得梯度',grads[0] is not None)
    c.ok('联合Critic获得梯度',grads[3] is not None)
    c.ok('联合旧策略无梯度',grads[1] is None)
    c.ok('联合固定优势无梯度',grads[2] is None)
    c.ok('联合旧critic值无梯度',grads[4] is None)
    c.ok('联合lambda目标无梯度',grads[5] is None)
    c.close('联合损失等于两边之和',parts['total_joint_loss'],float((a_loss+v_loss).detach()),atol=1e-6)
    c.ok('不被训练指标误写成基准成绩',0<=evaluate_policy(actor,critic,.95,6,20)['exact_success_within_limit']<=1)
    return {'count':len(c.names),'checks':c.names,
            'scope':'math_exact_enumeration_and_cpu_PPO_actor_critic_variants_not_pretrained_LLM_or_complete_PPO'}

In [21]:
fixed_checks=run_checks()
print('固定检查通过：',fixed_checks['count'],'项')
print('检查范围：',fixed_checks['scope'])
if RESULT_DIR is not None:
    save_json(RESULT_DIR/'checks.json',fixed_checks)

固定检查通过： 118 项
检查范围： math_exact_enumeration_and_cpu_PPO_actor_critic_variants_not_pretrained_LLM_or_complete_PPO


## 12. 三道独立练习

先补完再打开下面的 `RUN_EXERCISE_*`。默认关闭；执行 Notebook 全部单元成功并不表示练习已完成。

**练习 A：保守值裁剪**

给定 `prediction, old_value, target, epsilon`，计算 `0.5*mean(max(raw_squared, clipped_squared))`；`prediction` 需要梯度，旧值与目标不能接收梯度。正确处理目标在远处时的平坦段，以及新值偏离正确目标时的非平坦段。

In [22]:
def value_clip_ex_reference(pred,new_old,target,epsilon):
    """练习A：保守max版本。old/target只用作固定数值。"""
    require(type(epsilon) in (int,float) and math.isfinite(epsilon) and epsilon>0,'epsilon非法')
    old=new_old.detach();goal=target.detach()
    raw=(pred-goal).square()
    capped=old+(pred-old).clamp(-epsilon,epsilon)
    alternative=(capped-goal).square()
    return .5*torch.maximum(raw,alternative).mean()


def actor_entropy_ex_reference(new_logp,old_logp,advantages,log_dist,eps,entropy_coef):
    """练习B：明确把负熵纳入Actor；不会访问Critic。"""
    require(type(entropy_coef) in (int,float) and math.isfinite(entropy_coef)
            and entropy_coef>=0,'entropy_coef非法')
    actor,info=policy_loss(new_logp,old_logp,advantages,eps)
    require(log_dist.shape==(new_logp.numel(),2),'完整分布shape错误')
    entropy=-(log_dist.exp()*log_dist).sum(-1).mean()
    return actor-entropy_coef*entropy


def kl_gate_ex_reference(epochs_done,max_epochs,old_state_kl_value,target_kl,already_stopped):
    """练习C：Actor停止以后本rollout不重新启动；Critic可继续。"""
    require(type(epochs_done) is int and epochs_done>=0 and type(max_epochs) is int
            and max_epochs>0,'轮数非法')
    require(type(old_state_kl_value) in (float,int) and math.isfinite(old_state_kl_value)
            and old_state_kl_value>=0,'KL非有限或负数')
    require(type(target_kl) in (float,int) and math.isfinite(target_kl) and target_kl>0,
            'KL阈值非法')
    require(type(already_stopped) is bool,'停止标记必须为bool')
    if epochs_done>=max_epochs:
        return {'actor':False,'critic':False,'reason':'budget_exhausted'}
    if already_stopped or old_state_kl_value>target_kl:
        return {'actor':False,'critic':True,'reason':'actor_kl_stop'}
    return {'actor':True,'critic':True,'reason':'joint_update'}


def check_exercise_a(fn):
    c=Checks()
    for p,old,target in [(.5,0.,2.),(.5,0.,0.),(.1,0.,2.)]:
        x=torch.tensor([p],dtype=torch.float64,requires_grad=True)
        o=torch.tensor([old],dtype=torch.float64,requires_grad=True)
        y=torch.tensor([target],dtype=torch.float64,requires_grad=True)
        ans=fn(x,o,y,.2)
        wanted=value_clip_ex_reference(x,o,y,.2)
        c.close('A_loss'+str((p,target)),ans,wanted)
        ga,go,gy=torch.autograd.grad(ans,(x,o,y),allow_unused=True)
        expected=torch.autograd.grad(wanted,x)[0]
        c.close('A梯度'+str((p,target)),ga,expected)
        c.ok('A旧值与目标无grad'+str((p,target)),go is None and gy is None)
    c.raises('A_eps拒绝0',lambda:fn(torch.zeros(1),torch.zeros(1),torch.zeros(1),0.))
    return {'count':len(c.names),'passed':c.names}


def check_exercise_b(fn):
    c=Checks()
    z=torch.tensor([.4,-.2],dtype=torch.float64,requires_grad=True)
    logdist=F.log_softmax(torch.stack([z,-z],dim=-1),dim=-1)
    n=logdist[:,0]
    o=torch.tensor([-.69,-.69],dtype=torch.float64,requires_grad=True)
    a=torch.tensor([1.,-1.],dtype=torch.float64,requires_grad=True)
    result=fn(n,o,a,logdist,.2,.05)
    reference=actor_entropy_ex_reference(n,o,a,logdist,.2,.05)
    c.close('B_joint',result,reference)
    grad=torch.autograd.grad(result,(z,o,a),allow_unused=True,retain_graph=True)
    desired=torch.autograd.grad(reference,z)[0]
    c.close('B_Actor梯度',grad[0],desired)
    c.ok('B_旧策略和优势无梯度',grad[1] is None and grad[2] is None)
    c.raises('B_entcoef拒绝负数',lambda:fn(n,o,a,logdist,.2,-1.))
    return {'count':len(c.names),'passed':c.names}


def check_exercise_c(fn):
    c=Checks()
    assert_expected=[(0,4,0.,.02,False,{'actor':True,'critic':True,'reason':'joint_update'}),
                     (1,4,.02,.02,False,{'actor':True,'critic':True,'reason':'joint_update'}),
                     (1,4,.02001,.02,False,{'actor':False,'critic':True,'reason':'actor_kl_stop'}),
                     (2,4,0.,.02,True,{'actor':False,'critic':True,'reason':'actor_kl_stop'}),
                     (4,4,0.,.02,False,{'actor':False,'critic':False,'reason':'budget_exhausted'})]
    for num,(used,total,kl,limit,stopped,expected) in enumerate(assert_expected):
        c.ok('C_state_'+str(num),fn(used,total,kl,limit,stopped)==expected)
    c.raises('C_NaN',lambda:fn(1,4,float('nan'),.02,False))
    c.raises('C_bool停止标记',lambda:fn(1,4,0.,.02,1))
    c.raises('C_负轮数',lambda:fn(-1,4,0.,.02,False))
    return {'count':len(c.names),'passed':c.names}

In [23]:
RUN_EXERCISE_A=False

def value_clip_ex(prediction,old_value,target,epsilon):
    raise NotImplementedError('请自己实现保守max value loss')

if RUN_EXERCISE_A:
    print(check_exercise_a(value_clip_ex))

**练习 B：带熵正则的 Actor 目标**

输入当前/旧 log 概率、固定优势、**完整动作分布的 log 概率**、epsilon 和非负熵系数。

返回 `PPO policy loss - entropy_coef * mean_entropy`。不要丢掉 `min`；不能让旧策略或优势接收梯度；必须保持当前 Actor 的计算图。这个函数不涉及 Critic。

In [24]:
RUN_EXERCISE_B=False

def actor_entropy_ex(new_logp,old_logp,advantages,log_dist,eps,entropy_coef):
    raise NotImplementedError('请实现Actor PPO-Clip与熵正则')

if RUN_EXERCISE_B:
    print(check_exercise_b(actor_entropy_ex))

**练习 C：KL 停止门控**

输入已经使用的 epoch 数、最大 epoch 数、当前旧状态 KL、KL 阈值和当前 Actor 是否已停止，返回这轮是否允许 Actor、Critic 更新与原因。

- 如果 epoch 数已经耗尽，两者都不能更新。
- 如果 Actor 早已停止，或者 KL **严格超过**阈值，则只允许 Critic 更新。
- KL 恰等于阈值时仍允许联合更新；本批次一旦 Actor 停止，不再重新启动。

In [25]:
RUN_EXERCISE_C=False

def kl_gate_ex(epochs_done,max_epochs,old_state_kl_value,target_kl,already_stopped):
    raise NotImplementedError('请完成固定epoch与KL门控')

if RUN_EXERCISE_C:
    print(check_exercise_c(kl_gate_ex))

## 13. 参考实现、故障注入与验证

先完成独立练习，再阅读下一单元。它仅定义不同名称的参考实现，不会替你填入练习函数。

故障测试会有意尝试：取min代替max、混淆两种价值裁剪、让旧值反传、颠倒熵项符号、遗漏PPO的min、Actor KL停止时错误暂停Critic，以及丢掉停止状态。只有检查真正失败才记作检测到故障。

In [26]:
print('参考A:',check_exercise_a(value_clip_ex_reference)['count'],'项')
print('参考B:',check_exercise_b(actor_entropy_ex_reference)['count'],'项')
print('参考C:',check_exercise_c(kl_gate_ex_reference)['count'],'项')

参考A: 10 项
参考B: 4 项
参考C: 8 项


In [27]:
def mutation_checks():
    cases=[]
    def detects(name,probe):
        try:probe()
        except (AssertionError,LessonError,ValueError):
            cases.append({'fault':name,'caught':True})
        else:raise AssertionError('未发现故障：'+name)
    def wrong_min(x,old,target,eps):
        a=(x-target.detach()).square()
        b=(old.detach()+(x-old.detach()).clamp(-eps,eps)-target.detach()).square()
        return .5*torch.minimum(a,b).mean()
    detects('值裁剪max错写成min',lambda:check_exercise_a(wrong_min))
    def wrong_prediction(x,old,target,eps):
        return value_loss(x,old,target,eps,'clipped_pred')[0]
    detects('混淆保守max与clipped_pred',lambda:check_exercise_a(wrong_prediction))
    def wrong_old_grad(x,old,target,eps):
        return .5*torch.maximum((x-target).square(),
                 (old+(x-old).clamp(-eps,eps)-target).square()).mean()
    detects('旧值与固定目标接收梯度',lambda:check_exercise_a(wrong_old_grad))
    def wrong_sign(n,old,adv,dist,eps,coef):
        actor,_=policy_loss(n,old,adv,eps)
        entropy=-(dist.exp()*dist).sum(-1).mean()
        return actor+coef*entropy
    detects('熵正则符号反转',lambda:check_exercise_b(wrong_sign))
    def wrong_refresh(n,old,adv,dist,eps,coef):
        entropy=-(dist.exp()*dist).sum(-1).mean()
        r=(n-old)
        return -(r.exp()*adv).mean()-coef*entropy
    detects('PPO目标丢失clip',lambda:check_exercise_b(wrong_refresh))
    def wrong_hard_stop(used,total,kl,target,stopped):
        if used>=total or stopped or kl>target:
            return {'actor':False,'critic':False,'reason':'actor_kl_stop'}
        return {'actor':True,'critic':True,'reason':'joint_update'}
    detects('Actor KL停止错误阻断Critic',lambda:check_exercise_c(wrong_hard_stop))
    def wrong_epoch_boundary(used,total,kl,target,stopped):
        return kl_gate_ex_reference(used,total,kl,target,False)
    detects('KL停止标记不保留',lambda:check_exercise_c(wrong_epoch_boundary))
    return {'count':len(cases),'cases':cases,'scope':'targeted_function_mutations_not_formal_verification'}

In [28]:
faults=mutation_checks()
print('检测到的故障：',faults['count'])
for row in faults['cases']:
    print(' -',row['fault'])
if RESULT_DIR is not None:
    save_json(RESULT_DIR/'mutations.json',faults)

检测到的故障： 7
 - 值裁剪max错写成min
 - 混淆保守max与clipped_pred
 - 旧值与固定目标接收梯度
 - 熵正则符号反转
 - PPO目标丢失clip
 - Actor KL停止错误阻断Critic
 - KL停止标记不保留


## 14. 独立脚本的命令行

同名 `.py` 包含全部代码，运行不需要打开 Notebook：

```bash
python lesson_62_ppo_actor_critic_value_clipping.py --checks-only
python lesson_62_ppo_actor_critic_value_clipping.py --math-only
python lesson_62_ppo_actor_critic_value_clipping.py
python lesson_62_ppo_actor_critic_value_clipping.py --mode clipmax
```

还可在独立**开发**实验中修改：

```bash
python lesson_62_ppo_actor_critic_value_clipping.py --updates 30 --update-epochs 3 --clip-value 0.2 --target-state-kl 0.02
```

从已有运行目录在新进程重载：

```bash
python lesson_62_ppo_actor_critic_value_clipping.py --reload-run lesson_62_outputs/实际运行目录
```

更改奖励单位或大小时要重新考虑 `clip_value`，不能直接照搬数值。改变训练配置后，所有性能曲线和快照属于另一份实验。当前测试集已经公开，后续调优它只能被称作回归/开发，不能继续当作独立盲测。

In [29]:
def main(argv=None):
    parser=argparse.ArgumentParser(description=__doc__)
    modes=parser.add_mutually_exclusive_group()
    modes.add_argument('--checks-only',action='store_true')
    modes.add_argument('--math-only',action='store_true')
    modes.add_argument('--reload-run',type=Path)
    parser.add_argument('--mode',choices=['all','plain','clipmax','clipped_pred'],default='all')
    parser.add_argument('--updates',type=int,default=45)
    parser.add_argument('--episodes-per-update',type=int,default=24)
    parser.add_argument('--update-epochs',type=int,default=4)
    parser.add_argument('--target-state-kl',type=float,default=.04)
    parser.add_argument('--clip-value',type=float,default=.2)
    parser.add_argument('--output-dir',default='lesson_62_outputs')
    args=parser.parse_args(argv)
    torch.set_num_threads(1)
    if args.reload_run:
        print(json.dumps(reload_run(args.reload_run),ensure_ascii=False,indent=2));return 0
    if args.math_only:
        print(json.dumps({'value_clipping':value_clipping_examples(),
                          'gae_time_limit':exact_time_limit_bias()},ensure_ascii=False,indent=2))
        return 0
    checks=run_checks()
    print(f"固定检查通过：{checks['count']}项。三状态PPO Actor/Critic，不是大模型结果。")
    if args.checks_only:return 0
    cfg=Config(updates=args.updates,episodes_per_update=args.episodes_per_update,
               update_epochs=args.update_epochs,target_state_kl=args.target_state_kl,
               clip_value=args.clip_value)
    cfg.validate()
    out=experiment(cfg,args.output_dir,args.mode)
    save_json(out/'checks.json',checks)
    save_json(out/'mutations.json',mutation_checks())
    report=json.loads((out/'training.json').read_text(encoding='utf-8'))
    print('初始6步成功率：',report['baseline']['exact_success_within_limit'])
    for name,entry in report['arms'].items():
        print(name,'成功率',entry['final']['exact_success_within_limit'],
              'value RMSE',entry['final']['value_rmse'],
              'actor/critic更新',entry['actor_updates'],entry['critic_updates'])
    print('输出目录：',out.resolve())
    return 0

## 15. 本课验收

1. 为什么当前 Actor 对旧策略求概率比，而 Critic 要记住**旧价值**？
2. 为什么 `max(squared_errors)` 与 `MSE(clipped_prediction,target)` 不一样？
3. 为什么必须在第1次更新前计算并冻结 GAE？第3次 Critic 更新为何不再重新估值？
4. 外部截断为什么要 bootstrap，却不连接下一回合的优势？
5. KL 检查停止 Actor 时，为什么本课选择仍允许 Critic 继续更新？它保证什么、又不保证什么？
6. 为什么价值 RMSE 与六步精确成功率可能变化不同步？

**边界：** 这是一份离线3状态小模型、完整批次的序列/时间步两动作 PPO-Clip Actor 与可选值裁剪的教学集成。没有共享基座、minibatch、多进程采样、RNN、PPO value baseline 全套工具、安全强化学习或语言模型后训练；没有 CUDA/3060 性能测量。

**下一课：第63课——PPO 的 minibatch、梯度累积与 KL 诊断。** 深入研究同一批次如何打乱、组合不同长度回合，以及统计指标怎样避免被采样步数和停止次数误导。

## 原始资料

1. [PPO 原论文](https://arxiv.org/abs/1707.06347)：PPO-Clip 目标。
2. [Spinning Up：PPO](https://spinningup.openai.com/en/latest/algorithms/ppo.html)：优势、目标裁剪与 KL 早停。
3. [Spinning Up PyTorch PPO 实现](https://spinningup.openai.com/en/latest/_modules/spinup/algos/pytorch/ppo/ppo.html)：Rollout、GAE、Actor/Critic 独立优化器。
4. [Stable-Baselines3 PPO 文档](https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html)：价值函数可选裁剪与尺度依赖。
5. [SB3 PPO 源码](https://github.com/DLR-RM/stable-baselines3/blob/master/stable_baselines3/ppo/ppo.py)：`clipped_pred` 变体，不等于 `clipmax`。
6. [GAE 论文](https://arxiv.org/abs/1506.02438)：GAE、TD残差与 λ-return。
7. [Gymnasium 时间限制](https://gymnasium.farama.org/v1.1.1/tutorials/gymnasium_basics/handling_time_limits/)：真终止与外部截断。

本课的数学表格与训练结果均由代码计算；没有声称本机已运行大型预训练模型、GPU benchmark 或完整通用 PPO 训练框架。